In [ ]:
# [env] load .env (KEY=VALUE lines) into os.environ -- see .env.sample
# real env vars / Colab / Kaggle secrets take precedence; empty values are ignored
import os, pathlib
for _p in [pathlib.Path(".env"), pathlib.Path("../.env"),
           pathlib.Path("/content/drive/MyDrive/.env"),
           *sorted(pathlib.Path("/kaggle/input").glob("*/.env"))]:
    if _p.is_file():
        for _line in _p.read_text().splitlines():
            _k, _sep, _v = _line.partition("=")
            _k, _v = _k.strip(), _v.strip().strip("\"'")
            if _sep and _k and _v and not _k.startswith("#"):
                os.environ.setdefault(_k, _v)
        print("env loaded from", _p)
        break
else:
    print("no .env found -- copy .env.sample to .env and fill it in")

# SAR-to-Optical: complete pipeline notebook

**Every experiment runs as a plain Python function call in its own cell. No `!python`, no `%run`,
no command-line flags, no argparse anywhere in this notebook.** That whole class of bug (Colab's
kernel argv leaking into an argument parser) is structurally impossible here because there is no
argument parser to leak into.

## If a cell stops -- 24h Colab limit, a disconnect, anything
1. `Runtime > Restart session` (or start a fresh Colab session).
2. Re-run every cell from the top **down through the last `%%writefile` cell** (the setup +
   `common.py` + whichever other `.py` files precede where you were) -- these just rewrite files
   and take seconds; `prepare_data()` and `run_precheck()` print "already exists -- SKIPPING"
   and return immediately rather than redoing 20+ minutes of work.
3. Re-run the **exact same training cell** that was running when it stopped (e.g.
   `train_one("dualsar")`). It downloads its last checkpoint from the Hub and resumes from that
   step automatically -- same call, no arguments to change.
4. Continue down the notebook from there.

## "Run All" is safe and cheap to repeat
Every step in this notebook is idempotent: `prepare_data()`, `run_precheck()`, and `curate_data()`
each check the Hub first and skip straight to printing their existing result if already done.
`train_one(exp)` checks a few-hundred-byte `state.json` **before** touching the VAE, text
embeddings, dataset index, or the ~1.9GB of checkpoint weights -- so re-running Run All after
some experiments already finished costs about a second per finished experiment, not a
re-download, let alone re-training. Only the sanity checks (a few minutes) and
`evaluate_experiments()` (by design, so its numbers reflect whatever currently exists) always
re-run in full.

## Sanity checks -- placed where their prerequisites actually exist
Right after `run_precheck()`, `sanity_check_all()` runs 5 real training steps of the 6 **core**
experiment configs -- data loading, model forward/backward, loss finiteness, and a checkpoint
save/reload round-trip -- and prints PASS/FAIL for each, in under 5 minutes total.
**Do not skip it and do not proceed past a FAIL.** It exists specifically because a real bug
(the `wavelet` experiment's loss being silently ~6-7x larger in scale than every other
experiment's) previously ran undetected for 15,000 steps and several hours of GPU time; this
check catches it in about 10 seconds. That bug is fixed in `common.py` below.

`curated` is checked separately, right after `curate_data()` produces the file it needs
(`split_scene_curated.json`) -- checking it any earlier would fail on a missing prerequisite
regardless of anything else, since that file simply doesn't exist yet at that point.


In [1]:
# -- setup: run this once per fresh runtime --
!mkdir -p /content/sar_fyp
%cd /content/sar_fyp
!pip -q install kagglehub diffusers transformers accelerate safetensors torchmetrics[image] lpips sentence-transformers bitsandbytes peft

import os

from google.colab import userdata; os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_TOKEN")


/content/sar_fyp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 42.2 MB/s eta 0:00:00


## Shared code -- `common.py`
Everything downstream imports from this: paths, Hub sync, the dataset, model surgery, losses, metrics, checkpoint I/O, `train_one()`, and `sanity_check_all()`.

In [2]:
%%writefile common.py
"""
common.py — shared code for every SAR->optical experiment script.

Every experiment imports from here so that data loading, the model surgery,
the losses and the metrics are *byte-identical* across runs. That is what makes
the comparison table "same setting".

Dataset: "Sentinel-1&2 Image Pairs (SAR & Optical)" (Kaggle: requiemonk/
sentinel12-image-pairs-segregated-by-terrain) — 16,000 SEN1-2 pairs, 4 land-cover
classes (agri / barrenland / grassland / urban), 30 source scenes.
"""
import os, re, json, math, time, random, itertools, gc, shutil, resource
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# ───────────────────────────── paths / constants ─────────────────────────────
# HF_RUNS_REPO IS THE SOURCE OF TRUTH. Everything under CACHE is disposable —
# Colab wipes /content on every session end/disconnect, so nothing is allowed to
# live ONLY on local disk. Every script pulls what it needs from the Hub at
# startup and pushes what it produces before it can be lost.
CACHE = Path(os.environ.get("SAR_CACHE", "/content/sar_fyp_cache"))
DATA_META = CACHE / "data_meta"                               # splits, stats, latent scale (tiny files)
RUNS = CACHE / "runs"                                         # checkpoints (mirrored to the Hub)

KAGGLE_HANDLE = "requiemonk/sentinel12-image-pairs-segregated-by-terrain"
PATCH_SPLIT_URL = "https://raw.githubusercontent.com/yuuIind/SAR2Optical/main/data/split.json"

OLD_HF_REPO = os.environ["HF_REPO_ID"]          # your existing checkpoints (read-only)
OLD_VAE_TAG = "adapted16_final"
HF_RUNS_REPO = os.environ.get("HF_RUNS_REPO", OLD_HF_REPO)     # where new runs + data_meta live
HF_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")

DATA_META_FILES = ("index.json", "split_scene.json", "split_patch.json",
                   "split_leak_eval.json", "split_scene_curated.json", "color_stats.json",
                   "latent_scale.json", "precheck_results.json")


def require_hf_token():
    if not HF_TOKEN:
        raise RuntimeError(
            "HF_WRITE_TOKEN is not set. Nothing you produce in this cell will survive the session "
            "ending (Colab wipes /content). Set it before running anything else:\n"
            "  import os; os.environ['HF_WRITE_TOKEN'] = 'hf_...'\n"
            "and make sure the token has WRITE access to " + HF_RUNS_REPO)


def hf_push_file(local_path, path_in_repo, quiet=False):
    """Upload one file to HF_RUNS_REPO. Raises loudly on failure — a silent failure here is exactly
    the bug that lost your previous 00/01 outputs."""
    require_hf_token()
    from huggingface_hub import HfApi
    HfApi().upload_file(path_or_fileobj=str(local_path), path_in_repo=path_in_repo,
                        repo_id=HF_RUNS_REPO, repo_type="model", token=HF_TOKEN)
    if not quiet:
        log(f"  pushed -> hf://{HF_RUNS_REPO}/{path_in_repo}")


def hf_push_dir(local_dir, path_in_repo, quiet=False):
    require_hf_token()
    from huggingface_hub import HfApi
    HfApi().upload_folder(folder_path=str(local_dir), path_in_repo=path_in_repo,
                          repo_id=HF_RUNS_REPO, repo_type="model", token=HF_TOKEN)
    if not quiet:
        log(f"  pushed -> hf://{HF_RUNS_REPO}/{path_in_repo}/")


def hf_pull_file(path_in_repo, local_path, required=True):
    """Populate the local cache from the Hub. Returns True/False; raises only if required=True
    and the file truly isn't there (vs. already present locally, which is not re-downloaded)."""
    local_path = Path(local_path)
    if local_path.exists():
        return True
    from huggingface_hub import hf_hub_download
    import shutil
    try:
        p = hf_hub_download(HF_RUNS_REPO, path_in_repo, token=HF_TOKEN or None)
    except Exception as e:
        if required:
            raise FileNotFoundError(
                f"'{path_in_repo}' not found on hf://{HF_RUNS_REPO} and not cached locally at "
                f"{local_path}. ({e})\nIf this is data_meta, run 00_prepare_data.py (and "
                f"01_precheck_oracles.py for latent_scale.json) first — they push to the Hub "
                f"automatically. If this is a checkpoint, that experiment hasn't been trained "
                f"(or pushed) yet.")
        return False
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(p, local_path)
    return True


def hf_pull_dir(path_in_repo, local_dir, required=True):
    """Download every file under a prefix in HF_RUNS_REPO into local_dir (skips what's already there)."""
    from huggingface_hub import snapshot_download
    import shutil
    try:
        snap = snapshot_download(HF_RUNS_REPO, allow_patterns=[f"{path_in_repo}/*", f"{path_in_repo}/**/*"],
                                 token=HF_TOKEN or None)
    except Exception as e:
        if required:
            raise FileNotFoundError(f"'{path_in_repo}/' not found on hf://{HF_RUNS_REPO} ({e})")
        return False
    src = Path(snap) / path_in_repo
    if not src.exists():
        if required:
            raise FileNotFoundError(f"'{path_in_repo}/' not found on hf://{HF_RUNS_REPO}")
        return False
    Path(local_dir).mkdir(parents=True, exist_ok=True)
    for f in src.rglob("*"):
        if f.is_file():
            dst = Path(local_dir) / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(f, dst)
    log(f"pulled hf://{HF_RUNS_REPO}/{path_in_repo}/ -> {local_dir}")
    return True


def sync_data_meta():
    """Pull every data_meta file that exists on the Hub into the local cache. Call this at the
    top of every script — cheap (a few KB each) and makes a fresh session self-healing as long
    as 00/01 were ever run and pushed successfully."""
    DATA_META.mkdir(parents=True, exist_ok=True)
    got = []
    for f in DATA_META_FILES:
        if hf_pull_file(f"data_meta/{f}", DATA_META / f, required=False):
            got.append(f)
    log(f"data_meta synced from hf://{HF_RUNS_REPO}: {got or '(nothing found yet)'}")
    return got


def preflight():
    """Loud, immediate diagnostics. Called first thing in every script's main() so a bad Colab
    runtime type or missing prerequisites fail with a message you can't miss — not a 2-second
    silent exit under a collapsed '...' output."""
    log("=" * 70)
    log(f"PREFLIGHT  cache={CACHE}  hf_repo={HF_RUNS_REPO}  token_set={bool(HF_TOKEN)}")
    if not HF_TOKEN:
        log("  !! HF_WRITE_TOKEN is NOT set. Everything this session produces will be LOST when "
            "the runtime ends. Set os.environ['HF_WRITE_TOKEN'] before continuing.")
    try:
        import torch
        if not torch.cuda.is_available():
            log("  !! torch.cuda.is_available() is False. Runtime > Change runtime type > GPU.")
        else:
            log(f"  GPU: {torch.cuda.get_device_name(0)}  "
                f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
    except Exception as e:
        log(f"  !! torch import/CUDA check failed: {e}")
    log("=" * 70)

# runwayml/stable-diffusion-v1-5 was removed from the Hub in 2024; this is the official mirror.
SD_CANDIDATES = ("stable-diffusion-v1-5/stable-diffusion-v1-5", "runwayml/stable-diffusion-v1-5")

CLASSES = ("agri", "barrenland", "grassland", "urban")
CLASS_TEXT = {"agri": "agricultural", "barrenland": "barren", "grassland": "grassland", "urban": "urban"}
IMG = 256
TOTAL_CH = 16
FIXED_TB = 933
SEED = 42

# Every experiment is ONE change against 'baseline' (a controlled ablation). See train_one()'s
# docstring for what each one tests. 'curated' needs curate.py's curate_data() run first.
EXPS = {
    "baseline":   dict(split="scene",         sar_mode="gray3", cond="self", loss="std", model="regressor"),
    "patchsplit": dict(split="patch",         sar_mode="gray3", cond="self", loss="std", model="regressor"),
    "dualsar":    dict(split="scene",         sar_mode="dual",  cond="self", loss="std", model="regressor"),
    "wavelet":    dict(split="scene",         sar_mode="gray3", cond="self", loss="wavelet", model="regressor"),
    "sceneref":   dict(split="scene",         sar_mode="gray3", cond="ref",  loss="std", model="regressor"),
    "bridge":     dict(split="scene",         sar_mode="gray3", cond="self", loss="std", model="bridge"),
    "curated":    dict(split="scene_curated", sar_mode="gray3", cond="self", loss="std", model="regressor"),
}
DEFAULT_EXPS = ("baseline", "patchsplit", "dualsar", "wavelet", "sceneref", "bridge")
LOSS_W = dict(lat_shift=1.0, lat_l1=0.5, img_l1=1.0, ssim=0.5, mind=0.2, sam=0.1)
dev = torch.device("cuda")


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def sd_model_id():
    from huggingface_hub import model_info
    for mid in SD_CANDIDATES:
        try:
            model_info(mid); return mid
        except Exception:
            continue
    raise RuntimeError("Could not reach any Stable Diffusion 1.5 repo on the Hub.")


# ───────────────────────────── file naming ─────────────────────────────
_NAME = re.compile(r"(ROIs\d+)_([a-z]+)_s1_(\d+)_p(\d+)\.png$")


def parse_s1(relpath):
    """'agri/s1/ROIs1970_fall_s1_29_p59.png' -> dict(cls, season, scene, id)."""
    m = _NAME.search(relpath)
    if m is None:
        return None
    roi, season, sc, p = m.groups()
    cls = relpath.replace("\\", "/").split("/")[-3]
    return {"cls": cls, "season": season, "scene": f"{roi}_{season}_{sc}",
            "id": f"{roi}_{season}_{sc}_p{p}"}


def s2_of(s1_rel):
    return s1_rel.replace("/s1/", "/s2/").replace("_s1_", "_s2_")


# ───────────────────────────── leakage-free split ─────────────────────────────
def scene_split(items, val_frac=0.125, test_frac=0.125):
    """
    Split by SOURCE SCENE (no scene appears in two splits), per class, choosing the
    assignment whose val/test patch counts are closest to the targets.
    Each class keeps >=1 scene in val, >=1 in test, >=2 in train.
    Deterministic (exhaustive search, no RNG).
    """
    scene_cls, scene_n = {}, defaultdict(int)
    by_scene_cls = defaultdict(lambda: defaultdict(int))
    for it in items:
        scene_n[it["scene"]] += 1
        by_scene_cls[it["scene"]][it["cls"]] += 1
    for s, d in by_scene_cls.items():
        scene_cls[s] = max(d, key=d.get)                 # mixed scenes -> majority class
    assign = {}
    for c in CLASSES:
        sc = sorted([s for s in scene_cls if scene_cls[s] == c])
        N = sum(scene_n[s] for s in sc)
        best, best_cost = None, 1e18
        for combo in itertools.product((0, 1, 2), repeat=len(sc)):    # 0 train 1 val 2 test
            if combo.count(1) < 1 or combo.count(2) < 1 or combo.count(0) < 2:
                continue
            nv = sum(scene_n[s] for s, k in zip(sc, combo) if k == 1)
            nt = sum(scene_n[s] for s, k in zip(sc, combo) if k == 2)
            cost = abs(nv - val_frac * N) + abs(nt - test_frac * N)
            if cost < best_cost:
                best, best_cost = combo, cost
        for s, k in zip(sc, best):
            assign[s] = ("train", "val", "test")[k]
    out = {"train": [], "val": [], "test": []}
    for it in items:
        out[assign[it["scene"]]].append(it["id"])
    return out, assign


# ───────────────────────────── dataset ─────────────────────────────
def load_index():
    hf_pull_file("data_meta/index.json", DATA_META / "index.json")
    idx = json.load(open(DATA_META / "index.json"))
    root = Path(idx["root"])
    if not root.exists():
        # index.json was built in a PREVIOUS session; kagglehub's cache path is gone.
        # Re-download (kagglehub itself caches on disk, so this is idempotent within a session).
        import kagglehub
        new_root = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
        log(f"dataset root from a past session ({root}) is gone; re-downloaded to {new_root}")
        root = next((p for p in [new_root, *new_root.rglob('*')]
                    if p.is_dir() and all((p / c / 's1').is_dir() for c in CLASSES)), new_root)
    return root, {it["id"]: it for it in idx["items"]}


def load_split(name):
    hf_pull_file(f"data_meta/split_{name}.json", DATA_META / f"split_{name}.json")
    return json.load(open(DATA_META / f"split_{name}.json"))


def _load_gray(path):
    return torch.from_numpy(np.array(Image.open(path).convert("L"), np.float32) / 255.)


def _load_rgb(path):
    img = Image.open(path).convert("RGB")
    if img.size != (IMG, IMG):
        img = img.resize((IMG, IMG), Image.BILINEAR)
    return torch.from_numpy(np.array(img, np.float32) / 255.).permute(2, 0, 1)


def box_mean(x, k):
    """x: (H,W) in [0,1]. Spatial multilook (boxcar) with reflect padding."""
    p = k // 2
    return F.avg_pool2d(F.pad(x[None, None], (p, p, p, p), mode="reflect"), k, 1)[0, 0]


def sar_tensor(gray, mode):
    """
    gray3 : [raw, raw, raw]            (what your v3 was trained on)
    dual  : [raw, multilook5, multilook11] — ADDS a lower-variance mean-backscatter
            estimate without removing the speckle-bearing raw channel.
            NB: SEN1-2 PNGs are already log-scaled 8-bit, so this is multilook in the
            log domain (an approximation of intensity-domain multilooking).
    """
    if mode == "gray3":
        x = gray[None].repeat(3, 1, 1)
    elif mode == "dual":
        x = torch.stack([gray, box_mean(gray, 5), box_mean(gray, 11)])
    else:
        raise ValueError(mode)
    return x * 2 - 1


class PairDS(Dataset):
    def __init__(self, ids, root, index, sar_mode="gray3", augment=False, ref_rgb=None):
        self.ids, self.root, self.index = list(ids), Path(root), index
        self.sar_mode, self.augment, self.ref_rgb = sar_mode, augment, ref_rgb

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        it = self.index[self.ids[i]]
        sar = sar_tensor(_load_gray(self.root / it["s1"]), self.sar_mode)
        opt = _load_rgb(self.root / it["s2"]) * 2 - 1
        if self.augment:                               # paired geometric aug
            k = random.randint(0, 3)
            sar, opt = torch.rot90(sar, k, (1, 2)), torch.rot90(opt, k, (1, 2))
            if random.random() < 0.5:
                sar, opt = sar.flip(2), opt.flip(2)
        out = {"sar": sar, "opt": opt, "cls": it["cls"], "season": it["season"],
               "scene": it["scene"], "id": it["id"]}
        if self.ref_rgb is not None:
            c = torch.tensor(self.ref_rgb[it["id"]], dtype=torch.float32)
            out["ref"] = (c[:, None, None].expand(3, IMG, IMG) * 2 - 1).contiguous()
        return out


# ───────────────────────────── models ─────────────────────────────
class DetailEncoder(nn.Module):
    def __init__(s, ic=3, oc=12, nf=64):
        super().__init__()
        s.net = nn.Sequential(
            nn.Conv2d(ic, nf, 3, 2, 1), nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf * 2, 3, 2, 1), nn.GroupNorm(8, nf * 2), nn.SiLU(),
            nn.Conv2d(nf * 2, nf * 4, 3, 2, 1), nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, nf * 4, 3, 1, 1), nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, oc, 3, 1, 1))

    def forward(s, x):
        return s.net(x)


class AdaptedVAE(nn.Module):
    """Identical to the class in your final_eval_full.py."""
    def __init__(s, bv):
        super().__init__()
        s.base_vae, s.detail_enc = bv, DetailEncoder()
        o = bv.decoder.conv_in
        bv.decoder.conv_in = nn.Conv2d(TOTAL_CH, o.out_channels, o.kernel_size, o.stride,
                                       o.padding, bias=(o.bias is not None))

    @torch.no_grad()
    def enc(s, x):
        z = s.base_vae.encode(x).latent_dist.mode()
        return torch.cat([s.base_vae.post_quant_conv(z), s.detail_enc(x)], 1)

    def dec(s, l):                      # grad-enabled (used for joint decoder fine-tuning)
        return s.base_vae.decoder(l)


def load_vae(device, decoder_state=None):
    from diffusers import AutoencoderKL
    from huggingface_hub import hf_hub_download
    bv = AutoencoderKL.from_pretrained(sd_model_id(), subfolder="vae")
    v = AdaptedVAE(bv)
    v.base_vae.load_state_dict(torch.load(hf_hub_download(OLD_HF_REPO, f"{OLD_VAE_TAG}/base_vae.pth"),
                                          map_location="cpu"))
    v.detail_enc.load_state_dict(torch.load(hf_hub_download(OLD_HF_REPO, f"{OLD_VAE_TAG}/detail_encoder.pth"),
                                            map_location="cpu"))
    if decoder_state is not None:
        v.base_vae.decoder.load_state_dict(torch.load(decoder_state, map_location="cpu"))
    return v.to(device).float().eval().requires_grad_(False)


def build_unet(init_sd=True, variant="full"):
    """
    SD-1.5 UNet with 32-ch input / 16-ch output (same surgery as your v3).
    init_sd=True: SD weights, new conv_in/out initialised so that the pretrained
    4-channel pathway is preserved (not random) — used for all NEW full-size runs.
    variant="small": a BK-SDM-Small-style channel/depth reduction (half the block widths,
    1 resnet layer per block instead of 2) trained from scratch via distillation in
    09_distill_small_unet.py — init_sd is ignored for this variant since there is no
    pretrained checkpoint at this shape.
    """
    from diffusers import UNet2DConditionModel
    if variant == "full":
        unet = UNet2DConditionModel.from_pretrained(sd_model_id(), subfolder="unet")
    elif variant == "small":
        cfg = UNet2DConditionModel.load_config(sd_model_id(), subfolder="unet")
        cfg["block_out_channels"] = tuple(max(c // 2, 32) for c in cfg["block_out_channels"])
        cfg["layers_per_block"] = 1
        unet = UNet2DConditionModel.from_config(cfg)
        init_sd = False
    else:
        raise ValueError(f"unknown variant {variant}")
    oi, oo = unet.conv_in, unet.conv_out
    ni = nn.Conv2d(2 * TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    no = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    if init_sd:
        with torch.no_grad():
            ni.weight.normal_(0, 1e-3); ni.bias.copy_(oi.bias)
            ni.weight[:, 0:4] = 0.5 * oi.weight
            ni.weight[:, TOTAL_CH:TOTAL_CH + 4] = 0.5 * oi.weight
            no.weight.normal_(0, 1e-3); no.bias.zero_()
            no.weight[0:4] = oo.weight; no.bias[0:4] = oo.bias
    else:
        nn.init.normal_(ni.weight, 0, 0.02); nn.init.zeros_(ni.bias)
        nn.init.normal_(no.weight, 0, 0.02); nn.init.zeros_(no.bias)
    unet.conv_in, unet.conv_out = ni, no
    unet.config.in_channels, unet.config.out_channels = 2 * TOTAL_CH, TOTAL_CH
    return unet


def build_text_embeds(device, seasons=("fall", "summer")):
    from transformers import CLIPTextModel, CLIPTokenizer
    mid = sd_model_id()
    tok = CLIPTokenizer.from_pretrained(mid, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(mid, subfolder="text_encoder").to(device).eval()
    cache = {}

    @torch.no_grad()
    def emb(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    season_word = {"fall": "autumn", "summer": "summer", "spring": "spring", "winter": "winter"}
    for s in seasons:
        for c in CLASSES:
            cache[(s, c)] = emb(f"a {season_word[s]} satellite optical image of {CLASS_TEXT[c]} terrain")
    cache["null"] = emb("")
    del txt; torch.cuda.empty_cache()
    return cache


def get_cond(seasons, classes, cache, drop_p=0.0):
    out = []
    for s, c in zip(seasons, classes):
        out.append(cache["null"] if random.random() < drop_p else cache.get((s, c), cache["null"]))
    return torch.stack(out)


# ───────────────────────────── ResShift schedule ─────────────────────────────
def resshift_eta(T=15, eta_1=0.04, eta_T=0.99):
    i = np.arange(T)
    return torch.tensor(eta_1 * (eta_T / eta_1) ** (i / (T - 1)), dtype=torch.float32)


@torch.no_grad()
def resshift_sample(m, y, cond, eta, kappa, T, gen=None):
    B = y.shape[0]
    x = y + kappa * torch.sqrt(eta[-1]) * torch.randn(y.shape, generator=gen, device=y.device)
    for t in reversed(range(T)):
        tb = torch.full((B,), int(t * 1000 / T), device=y.device, dtype=torch.long)
        x0 = m(torch.cat([x, y], 1), tb, encoder_hidden_states=cond).sample
        if t > 0:
            ep = eta[t - 1]
            x = x0 + ep * (y - x0) + kappa * torch.sqrt(ep) * torch.randn(y.shape, generator=gen, device=y.device)
        else:
            x = x0
    return x


# ───────────────────────────── losses ─────────────────────────────
def shift_tolerant_l1(pred, tgt, r=1):
    """Per-sample min over integer shifts in [-r, r]^2 (latent cells)."""
    H, W = pred.shape[-2:]
    p = pred[..., r:H - r, r:W - r]
    errs = []
    for dy in range(-r, r + 1):
        for dx in range(-r, r + 1):
            t = tgt[..., r + dy:H - r + dy, r + dx:W - r + dx]
            errs.append((p - t).abs().flatten(1).mean(1))
    return torch.stack(errs, 1).min(1).values.mean()


def ssim_loss(p, g):
    from torchmetrics.functional.image import structural_similarity_index_measure as ssim
    return 1 - ssim(p.float(), g.float(), data_range=2.0)


def _gauss_blur(x, sigma=1.0):
    k = int(2 * math.ceil(2 * sigma) + 1)
    ax = torch.arange(k, device=x.device, dtype=x.dtype) - k // 2
    g = torch.exp(-ax ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    C = x.shape[1]
    x = F.conv2d(F.pad(x, (k // 2, k // 2, 0, 0), mode="reflect"), g.view(1, 1, 1, k).repeat(C, 1, 1, 1), groups=C)
    return F.conv2d(F.pad(x, (0, 0, k // 2, k // 2), mode="reflect"), g.view(1, 1, k, 1).repeat(C, 1, 1, 1), groups=C)


def mind(x, sigma=1.0):
    """Light MIND descriptor on a 1-channel image (B,1,H,W): 4-neighbour self-similarity."""
    shifts = ((0, 2), (2, 0), (0, -2), (-2, 0))
    D = [_gauss_blur((x - torch.roll(x, s, (2, 3))) ** 2, sigma) for s in shifts]
    D = torch.cat(D, 1)
    V = D.mean(1, keepdim=True).clamp_min(1e-4)
    return torch.exp(-D / V)


def mind_loss(pred_rgb, sar):
    gp = pred_rgb.mean(1, keepdim=True)
    gs = sar[:, :1]
    return (mind(gp) - mind(gs)).abs().mean()


def sam_loss(p, g):
    p = ((p + 1) / 2).clamp(1e-4, 1); g = ((g + 1) / 2).clamp(1e-4, 1)
    c = (p * g).sum(1) / (p.norm(dim=1) * g.norm(dim=1) + 1e-8)
    return torch.acos(c.clamp(-1 + 1e-6, 1 - 1e-6)).mean()


def haar(x):
    """Single-level orthonormal Haar DWT -> (LL, LH, HL, HH), each half resolution."""
    a, b = x[..., 0::2, 0::2], x[..., 0::2, 1::2]
    c, d = x[..., 1::2, 0::2], x[..., 1::2, 1::2]
    return (a + b + c + d) / 2, (a - b + c - d) / 2, (a + b - c - d) / 2, (a - b - c + d) / 2


def wavelet_l1(p, g, w_ll=2.0, w_hi=1.0, levels=2):
    """
    Frequency-decoupled L1, weighted AVERAGE (not sum). An earlier version summed each band's
    L1 mean, which made this term's magnitude ~6-7x larger than a plain image L1 for any other
    experiment (visible as e.g. img_l1~2.7-4 for 'wavelet' vs ~0.2-0.3 for 'baseline' in the
    logs) -- silently changing the effective learning rate for that one experiment and making it
    not a controlled ablation. sanity_check_all() checks final losses are the same order of
    magnitude across experiments specifically to catch this class of bug going forward.
    """
    num, den = 0.0, 0.0
    pp, gg = p, g
    for _ in range(levels):
        pl, *ph = haar(pp); gl, *gh = haar(gg)
        for a, b in zip(ph, gh):
            num = num + w_hi * (a - b).abs().mean()
            den = den + w_hi
        pp, gg = pl, gl
    num = num + w_ll * (pp - gg).abs().mean()
    den = den + w_ll
    return num / den


# ───────────────────────────── metrics ─────────────────────────────
def to01(t):
    return (t.clamp(-1, 1) + 1) / 2


def psnr(p, g):                     # inputs in [-1,1] — identical to your eval (peak-to-peak 2)
    return float(10 * np.log10(4.0 / max(F.mse_loss(p, g).item(), 1e-12)))


def sam_deg(p, g):
    p = to01(p).clamp(1e-4, 1).flatten(2).transpose(1, 2)
    g = to01(g).clamp(1e-4, 1).flatten(2).transpose(1, 2)
    c = ((p * g).sum(-1) / (p.norm(dim=-1) * g.norm(dim=-1) + 1e-8)).clamp(-1, 1)
    return torch.rad2deg(torch.acos(c)).mean().item()


def ergas(p, g):
    """
    FIXED. Your previous version took RMSE in [-1,1] units but the band mean in [0,1],
    which inflates ERGAS by exactly 2x. Here both are in [0,1]; ratio h/l = 1 (same GSD).
    """
    p, g = to01(p), to01(g)
    e = 0.0
    for b in range(3):
        rmse = torch.sqrt(F.mse_loss(p[:, b], g[:, b]))
        e += (rmse / (g[:, b].mean() + 1e-8)) ** 2
    return float(100 * torch.sqrt(e / 3))


def cc(p, g):
    t = 0.0
    for b in range(3):
        x = p[:, b].flatten() - p[:, b].mean(); y = g[:, b].flatten() - g[:, b].mean()
        t += float((x * y).sum() / (x.norm() * y.norm() + 1e-8))
    return t / 3


def scene_bootstrap_ci(values, scenes, n_boot=2000, seed=0):
    """95% CI that resamples SCENES (the real unit of independence), not patches."""
    rng = np.random.default_rng(seed)
    values, scenes = np.asarray(values, float), np.asarray(scenes)
    us = np.unique(scenes)
    groups = [values[scenes == s] for s in us]
    stats = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(us), len(us))
        stats.append(np.concatenate([groups[k] for k in pick]).mean())
    return float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))


# ───────────────────────────── checkpoint I/O ─────────────────────────────
def save_run(run_dir, unet, vae=None, cfg=None, tag="best", push=True, exp=None):
    """Writes locally AND (by default) pushes to the Hub immediately. push=False is only for the
    high-frequency local-only autosave inside the training loop (see 02_train.py); every point
    that could be the LAST thing written before a disconnect must push=True."""
    from safetensors.torch import save_file
    d = Path(run_dir) / tag
    d.mkdir(parents=True, exist_ok=True)
    save_file({k: v.detach().to(torch.float16).cpu().contiguous() for k, v in unet.state_dict().items()},
              str(d / "unet.safetensors"))
    if vae is not None:
        torch.save(vae.base_vae.decoder.state_dict(), d / "decoder.pth")
    if cfg is not None:
        json.dump(cfg, open(d / "config.json", "w"), indent=2)
    if push and exp is not None:
        hf_push_dir(d, f"fyp_runs/{exp}/{tag}")


def load_run(run_dir, device, tag="best", exp=None):
    from safetensors.torch import load_file
    d = Path(run_dir) / tag
    if exp is not None:                                       # pull from the Hub if not cached locally
        for f in ("config.json", "unet.safetensors", "decoder.pth"):
            hf_pull_file(f"fyp_runs/{exp}/{tag}/{f}", d / f, required=(f != "decoder.pth"))
    cfg = json.load(open(d / "config.json"))
    unet = build_unet(init_sd=False, variant=cfg.get("arch", "full"))
    unet.load_state_dict({k: v.float() for k, v in load_file(str(d / "unet.safetensors")).items()})
    dec = d / "decoder.pth"
    vae = load_vae(device, decoder_state=str(dec) if dec.exists() else None)
    return unet.to(device).eval().requires_grad_(False), vae, cfg


def push_to_hf(local_dir, path_in_repo):
    """Kept for backward compatibility with earlier drafts of the scripts; prefer save_run(push=True)."""
    hf_push_dir(local_dir, path_in_repo)


# ───────────────────────────── training: ONE function, no argparse, no subprocess ─────────────────────────────
@torch.no_grad()
def quick_val(unet, vae, emb, dl_val, scale, tb, cfg, n_batches=16):
    unet.eval(); ps = []
    eta = resshift_eta().to(dev)
    g = torch.Generator(device=dev).manual_seed(0)
    for k, b in enumerate(dl_val):
        if k >= n_batches: break
        sar, opt = b["sar"].to(dev), b["opt"].to(dev)
        y = vae.enc(sar) * scale
        c = get_cond(b["season"], b["cls"], emb).to(dev)
        if cfg["model"] == "bridge":
            lat = resshift_sample(unet, y, c, eta, 2.0, 15, gen=g)
        else:
            second = vae.enc(b["ref"].to(dev)) * scale if cfg["cond"] == "ref" else y
            tbv = torch.full((y.shape[0],), tb, device=dev, dtype=torch.long)
            lat = unet(torch.cat([y, second], 1), tbv, encoder_hidden_states=c).sample
        rec = vae.dec(lat / scale).clamp(-1, 1)
        ps += [psnr(rec[j:j + 1], opt[j:j + 1]) for j in range(rec.shape[0])]
    unet.train()
    return float(np.mean(ps))


def train_one(exp, steps=15000, bs=8, lr=5e-5, dec_lr=1e-5, no_dec_ft=False, checkpoint_every=1000,
              val_every=1000, max_hours=23.0, seed=42, adam8bit=False, vae=None, emb=None):
    """
    Call this directly -- ONE experiment, ONE function call, no argparse, no subprocess, no
    command-line flags to get wrong. This is the entire recommended workflow:

        train_one("baseline")

    Resumable by construction: every call first checks hf://HF_RUNS_REPO/fyp_runs/<exp>/last/ and
    picks up from there if it exists. If a cell is killed for ANY reason -- Colab's 24h limit, a
    disconnect, anything -- you do not diagnose it. You just re-run the SAME cell (same call,
    same arguments) in a new session; it downloads its last checkpoint and continues from that
    step. checkpoint_every defaults to 1000 (tighter than earlier drafts) specifically so a kill
    at a random, unpredictable moment costs at most ~1000 steps of re-work, not more.

    "Run All" on the whole notebook is safe to do repeatedly: an experiment that's already
    finished (step >= steps) is detected from a few-hundred-byte state.json read, before the VAE,
    text embeddings, dataset index, or the ~1.9GB of checkpoint weights are ever touched -- so
    re-running a finished experiment's cell costs about a second, not a download.

    Every experiment is ONE change against 'baseline':
      baseline    per-scene split, [raw,raw,raw] SAR, input [z_sar || z_sar], std loss
      patchsplit  identical, but trained on the published per-PATCH split  -> leakage size
      dualsar     SAR input [raw, multilook5, multilook11] (additive, not despeckling)
      wavelet     image L1 replaced by 2-level Haar L1 with 2x weight on LL (low-freq)
      sceneref    UNet second input = latent of the scene's reference colour (leave-one-out
                  mean of other patches) instead of a z_sar copy
      bridge      the ResShift stochastic branch (T=15, kappa=2)
      curated     needs curate_data() (curate.py) run first -- trains on the filtered split

    Pass `vae`/`emb` (already loaded) to reuse them across several train_one() calls in the same
    session instead of reloading the ~2GB VAE every time; if you do, note the decoder is reset to
    its pristine (pre-fine-tune) weights at the start of every call, so experiment 2 never
    silently inherits experiment 1's already-adapted decoder.
    """
    assert exp in EXPS, f"unknown exp {exp!r}; choices are {list(EXPS)}"
    cfg = dict(EXPS[exp]); seed_all(seed)
    t0 = time.time()
    deadline = t0 + max_hours * 3600
    run_dir = RUNS / exp; run_dir.mkdir(parents=True, exist_ok=True)

    # ── cheap early-exit, BEFORE touching the VAE, text embeddings, dataset index, or checkpoint
    # weights: just read state.json (a few hundred bytes). An earlier version did this check only
    # after loading ~2GB of VAE/embeddings and downloading ~1.9GB of checkpoint weights, so
    # re-running an already-finished experiment (exactly what happens on "Run All" once some
    # experiments are done) cost close to a minute of pure download for nothing. ──
    state_f = run_dir / "last" / "state.json"
    hf_pull_file(f"fyp_runs/{exp}/last/state.json", state_f, required=False)
    step, best = 0, -1.0
    if state_f.exists():
        st = json.load(open(state_f)); step, best = st["step"], st["best"]
    if step >= steps:
        log(f"[{exp}] already finished (step {step} >= steps={steps}) -- skipping immediately, "
            f"nothing downloaded or loaded.")
        return "already_done"

    ls_path = DATA_META / "latent_scale.json"
    if not ls_path.exists():
        hf_pull_file("data_meta/latent_scale.json", ls_path, required=True)
    ls = json.load(open(ls_path))
    scale, tb = float(ls["latent_scale"]), int(ls["fixed_tb"])

    root, index = load_index()
    sp = load_split(cfg["split"])
    ref = json.load(open(DATA_META / "color_stats.json"))["scene_loo_rgb"] if cfg["cond"] == "ref" else None
    ds_tr = PairDS(sp["train"], root, index, sar_mode=cfg["sar_mode"], augment=True, ref_rgb=ref)
    ds_va = PairDS(sp["val"], root, index, sar_mode=cfg["sar_mode"], ref_rgb=ref)
    dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True, num_workers=4, drop_last=True,
                       pin_memory=True, persistent_workers=False)
    dl_va = DataLoader(ds_va, batch_size=8, shuffle=False, num_workers=2)
    log(f"[{exp}] cfg={cfg}  train={len(ds_tr):,} val={len(ds_va):,}")

    own_vae = vae is None
    if own_vae:
        vae = load_vae(dev)
    dec_ft = (cfg["model"] == "regressor") and not no_dec_ft
    if dec_ft:
        if not hasattr(vae, "_pristine_decoder_state"):
            vae._pristine_decoder_state = {k: v.clone().cpu() for k, v in vae.base_vae.decoder.state_dict().items()}
        vae.base_vae.decoder.load_state_dict(vae._pristine_decoder_state)
    own_emb = emb is None
    if own_emb:
        emb = build_text_embeds(dev)

    unet = opt = it = None
    status = "done"
    try:
        unet = build_unet(init_sd=True)
        if state_f.exists():
            from safetensors.torch import load_file
            hf_pull_file(f"fyp_runs/{exp}/last/unet.safetensors", run_dir / "last" / "unet.safetensors")
            hf_pull_file(f"fyp_runs/{exp}/last/decoder.pth", run_dir / "last" / "decoder.pth", required=False)
            unet.load_state_dict({k: v.float() for k, v in load_file(str(run_dir / "last" / "unet.safetensors")).items()})
            if dec_ft and (run_dir / "last" / "decoder.pth").exists():
                vae.base_vae.decoder.load_state_dict(torch.load(run_dir / "last" / "decoder.pth", map_location="cpu"))
            log(f"[{exp}] RESUMING from step {step} (best val PSNR so far {best:.3f})")
        else:
            log(f"[{exp}] STARTING from scratch (no checkpoint on hf://{HF_RUNS_REPO}/fyp_runs/{exp}/last/)")
        unet = unet.to(dev).train()
        unet.enable_gradient_checkpointing()
        vae.base_vae.enable_gradient_checkpointing()
        vae.base_vae.decoder.train()

        params = [{"params": unet.parameters(), "lr": lr}]
        if dec_ft:
            vae.base_vae.decoder.requires_grad_(True).train()
            params.append({"params": vae.base_vae.decoder.parameters(), "lr": dec_lr})
        if adam8bit:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit(params, weight_decay=1e-2)
        else:
            opt = torch.optim.AdamW(params, weight_decay=1e-2)
        base_lrs = [g["lr"] for g in opt.param_groups]
        eta = resshift_eta().to(dev); kappa, T = 2.0, 15

        run_cfg = {**cfg, "exp": exp, "latent_scale": scale, "fixed_tb": tb, "T": T, "kappa": kappa,
                   "eta_1": 0.04, "eta_T": 0.99, "loss_w": LOSS_W, "steps": steps, "bs": bs, "lr": lr,
                   "dec_ft": dec_ft}
        warm, start_step = 300, step
        it = iter(dl_tr)
        log(f"[{exp}] >>> TRAINING LOOP ENTERED, step {step} -> {steps} <<<")
        while step < steps:
            try: b = next(it)
            except StopIteration: it = iter(dl_tr); b = next(it)
            f = min(1.0, (step - start_step + 1) / warm) * 0.5 * (1 + math.cos(math.pi * step / steps))
            for g, lr0 in zip(opt.param_groups, base_lrs): g["lr"] = lr0 * max(f, 0.02)

            sar, optimg = b["sar"].to(dev, non_blocking=True), b["opt"].to(dev, non_blocking=True)
            with torch.no_grad():
                y = vae.enc(sar) * scale
                x0 = vae.enc(optimg) * scale
                second = vae.enc(b["ref"].to(dev)) * scale if cfg["cond"] == "ref" else y
            c = get_cond(b["season"], b["cls"], emb, drop_p=0.1 if cfg["model"] == "bridge" else 0.0).to(dev)

            with torch.autocast("cuda", dtype=torch.bfloat16):
                if cfg["model"] == "bridge":
                    t = torch.randint(0, T, (y.shape[0],), device=dev)
                    et = eta[t].view(-1, 1, 1, 1)
                    xt = x0 + et * (y - x0) + kappa * et.sqrt() * torch.randn_like(x0)
                    pred = unet(torch.cat([xt, y], 1), (t * 1000 // T).long(), encoder_hidden_states=c).sample.float()
                    loss = F.mse_loss(pred, x0) + 0.1 * (pred - x0).abs().mean()
                    parts = {"mse": loss.item()}
                else:
                    tbv = torch.full((y.shape[0],), tb, device=dev, dtype=torch.long)
                    pred = unet(torch.cat([y, second], 1), tbv, encoder_hidden_states=c).sample.float()
                    rec = vae.dec(pred / scale).float().clamp(-1, 1)
                    img_term = wavelet_l1(rec, optimg) if cfg["loss"] == "wavelet" else (rec - optimg).abs().mean()
                    parts = {"lat_shift": shift_tolerant_l1(pred, x0), "lat_l1": (pred - x0).abs().mean(),
                             "img_l1": img_term, "ssim": ssim_loss(rec, optimg),
                             "mind": mind_loss(rec, sar), "sam": sam_loss(rec, optimg)}
                    loss = sum(LOSS_W[k] * v for k, v in parts.items())
                    parts = {k: float(v) for k, v in parts.items()}
            assert torch.isfinite(loss), f"[{exp}] non-finite loss at step {step}: {loss}"
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
            opt.step(); step += 1

            if step % 100 == 0:
                el = (time.time() - t0) / 3600
                log(f"[{exp}] step {step}/{steps} loss {loss.item():.4f} " +
                    " ".join(f"{k}={v:.3f}" for k, v in parts.items()) + f" | {el:.2f} h")
            if step % 500 == 0:
                ram_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
                log(f"[{exp}]   mem: RAM(peak-so-far) {ram_gb:.2f} GB  "
                    f"VRAM(alloc) {torch.cuda.memory_allocated() / 1e9:.2f} GB  "
                    f"VRAM(reserved) {torch.cuda.memory_reserved() / 1e9:.2f} GB")
            if step % val_every == 0 or step == steps:
                v = quick_val(unet, vae, emb, dl_va, scale, tb, cfg)
                log(f"[{exp}]   val PSNR (128 imgs) {v:.3f}  best {max(best, v):.3f}")
                if v > best:
                    best = v
                    save_run(run_dir, unet, vae if dec_ft else None, {**run_cfg, "best_step": step,
                             "val_psnr_128": v}, tag="best", push=True, exp=exp)
            timeout = time.time() >= deadline
            if step % checkpoint_every == 0 or step == steps or timeout:
                save_run(run_dir, unet, vae if dec_ft else None, run_cfg, tag="last", push=True, exp=exp)
                json.dump({"step": step, "best": best}, open(run_dir / "last" / "state.json", "w"))
                hf_push_file(run_dir / "last" / "state.json", f"fyp_runs/{exp}/last/state.json", quiet=True)
            if timeout:
                log(f"[{exp}] reached max_hours={max_hours} at step {step}. 'last/' is pushed. "
                    f"Re-run train_one({exp!r}) in a new session to resume.")
                status = "timeout"
                break
        else:
            log(f"[{exp}] DONE: best val PSNR (128 imgs) {best:.3f}  "
                f"(checkpoints on hf://{HF_RUNS_REPO}/fyp_runs/{exp}/)")
        return status
    finally:
        if unet is not None: del unet
        if opt is not None: del opt
        if it is not None: del it
        del dl_tr, dl_va
        if own_vae and vae is not None: del vae
        if own_emb and emb is not None: del emb
        gc.collect()
        torch.cuda.empty_cache()


# ───────────────────────────── sanity checks: run BEFORE spending real GPU-hours ─────────────────────────────
def sanity_check(exp, n_steps=5, bs=4):
    """
    Runs a MINIATURE version of train_one -- n_steps steps, no Hub push -- to confirm the whole
    pipeline for this exp actually works: data loads, the model builds, forward/backward produces
    a finite loss and finite gradients, and a checkpoint saved locally can be reloaded and matches.
    Takes seconds, not hours. This is exactly the kind of check that would have caught the
    wavelet_l1 scale bug before it silently ran for 15,000 steps.

    Returns a report dict; never raises -- check report["ok"] and report["error"].
    """
    assert exp in EXPS, f"unknown exp {exp!r}"
    cfg = dict(EXPS[exp])
    report = {"exp": exp, "ok": False, "steps_run": 0, "losses": [], "checkpoint_roundtrip_ok": False,
             "error": None}
    unet = opt = vae = emb = dl_tr = it = None
    try:
        seed_all(0)
        ls_path = DATA_META / "latent_scale.json"
        assert ls_path.exists(), "latent_scale.json missing -- run run_precheck() first"
        ls = json.load(open(ls_path)); scale, tb = float(ls["latent_scale"]), int(ls["fixed_tb"])

        root, index = load_index()
        sp = load_split(cfg["split"])
        assert len(sp["train"]) > 0 and len(sp["val"]) > 0, f"empty split for exp={exp!r}"
        ref = json.load(open(DATA_META / "color_stats.json"))["scene_loo_rgb"] if cfg["cond"] == "ref" else None
        ds_tr = PairDS(sp["train"][:64], root, index, sar_mode=cfg["sar_mode"], augment=True, ref_rgb=ref)
        dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True, num_workers=0)

        vae = load_vae(dev)
        dec_ft = cfg["model"] == "regressor"
        if dec_ft: vae.base_vae.decoder.requires_grad_(True).train()
        emb = build_text_embeds(dev)
        unet = build_unet(init_sd=True).to(dev).train()
        params = [{"params": unet.parameters(), "lr": 5e-5}]
        if dec_ft: params.append({"params": vae.base_vae.decoder.parameters(), "lr": 1e-5})
        opt = torch.optim.AdamW(params)
        eta = resshift_eta().to(dev); kappa, T = 2.0, 15

        it = iter(dl_tr)
        for step in range(n_steps):
            try: b = next(it)
            except StopIteration: it = iter(dl_tr); b = next(it)
            sar, optimg = b["sar"].to(dev), b["opt"].to(dev)
            with torch.no_grad():
                y = vae.enc(sar) * scale
                x0 = vae.enc(optimg) * scale
                second = vae.enc(b["ref"].to(dev)) * scale if cfg["cond"] == "ref" else y
            c = get_cond(b["season"], b["cls"], emb, drop_p=0.1 if cfg["model"] == "bridge" else 0.0).to(dev)
            if cfg["model"] == "bridge":
                t = torch.randint(0, T, (y.shape[0],), device=dev)
                et = eta[t].view(-1, 1, 1, 1)
                xt = x0 + et * (y - x0) + kappa * et.sqrt() * torch.randn_like(x0)
                pred = unet(torch.cat([xt, y], 1), (t * 1000 // T).long(), encoder_hidden_states=c).sample
                loss = F.mse_loss(pred, x0)
            else:
                tbv = torch.full((y.shape[0],), tb, device=dev, dtype=torch.long)
                pred = unet(torch.cat([y, second], 1), tbv, encoder_hidden_states=c).sample
                rec = vae.dec(pred / scale).clamp(-1, 1)
                img_term = wavelet_l1(rec, optimg) if cfg["loss"] == "wavelet" else (rec - optimg).abs().mean()
                loss = (shift_tolerant_l1(pred, x0) + 0.5 * (pred - x0).abs().mean() + img_term
                       + 0.5 * ssim_loss(rec, optimg) + 0.2 * mind_loss(rec, sar) + 0.1 * sam_loss(rec, optimg))
            assert torch.isfinite(loss), f"non-finite loss at step {step}: {loss.item()}"
            opt.zero_grad(); loss.backward()
            for p in unet.parameters():
                if p.grad is not None:
                    assert torch.isfinite(p.grad).all(), "non-finite gradient"
                    break
            opt.step()
            report["losses"].append(round(float(loss.item()), 4))
            report["steps_run"] = step + 1

        tmp_dir = Path("/content/_sanity_tmp") / exp
        shutil.rmtree(tmp_dir, ignore_errors=True)
        save_run(tmp_dir, unet, vae if dec_ft else None, {**cfg, "latent_scale": scale, "fixed_tb": tb},
                 tag="test", push=False, exp=None)
        unet2, vae2, cfg2 = load_run(tmp_dir, dev, tag="test", exp=None)
        k0 = next(iter(unet.state_dict()))
        assert torch.allclose(unet.state_dict()[k0].float().cpu(), unet2.state_dict()[k0].float().cpu(),
                              atol=1e-2), "checkpoint round-trip mismatch"
        report["checkpoint_roundtrip_ok"] = True
        shutil.rmtree(tmp_dir, ignore_errors=True)
        del unet2, vae2

        report["ok"] = True
    except Exception as e:
        report["error"] = f"{type(e).__name__}: {e}"
    finally:
        # unet/opt/vae/emb/dl_tr/it are locals -- they're freed automatically when this function
        # returns; gc.collect()+empty_cache() just makes CUDA release that memory immediately
        # rather than lazily, which matters if the caller runs several of these back to back.
        gc.collect(); torch.cuda.empty_cache()
    return report


def sanity_check_all(exps=None, n_steps=5):
    """
    Run sanity_check() for every experiment config and print a clear PASS/FAIL table. Also flags
    if any two experiments' loss magnitudes differ by >3x at step 0 -- a real, cheap check that
    would have caught the wavelet_l1 scale bug (its img_l1 term was ~6-7x every other experiment's)
    before 15,000 steps and hours of GPU time were spent on a miscalibrated loss.

    Call this ONCE, right after run_precheck(), before any train_one() cell.
    """
    exps = exps or list(EXPS)
    log("=" * 70)
    log(f"SANITY CHECK: validating {exps} end-to-end ({n_steps} steps each, no Hub push)")
    log("=" * 70)
    results = {}
    for exp in exps:
        log(f"  checking [{exp}] ...")
        r = sanity_check(exp, n_steps=n_steps)
        results[exp] = r
        status = "PASS" if r["ok"] else "FAIL"
        log(f"  [{exp}] {status}  losses={r['losses']}  ckpt_roundtrip={r['checkpoint_roundtrip_ok']}" +
            (f"\n    ERROR: {r['error']}" if r["error"] else ""))

    first_losses = {e: r["losses"][0] for e, r in results.items() if r["ok"] and r["losses"]}
    if len(first_losses) > 1:
        lo, hi = min(first_losses.values()), max(first_losses.values())
        if hi > 3 * max(lo, 1e-6):
            log(f"  !! WARNING: step-0 loss magnitudes range from {lo:.3f} to {hi:.3f} (>3x) across "
                f"experiments: {first_losses}. That usually means one experiment's loss is scaled "
                f"differently and isn't a controlled ablation against the others -- check it before "
                f"training for real.")

    log("=" * 70)
    failed = [e for e, r in results.items() if not r["ok"]]
    if failed:
        log(f"SANITY CHECK FAILED for: {failed} -- fix these before running the real training cells.")
    else:
        log("ALL SANITY CHECKS PASSED. Safe to run the training cells below.")
    return results


Writing common.py


## Step 0 -- prepare the dataset (idempotent: skips if already done)

In [3]:
%%writefile data_prep.py
"""
data_prep.py — call prepare_data() directly. No argparse, no __main__ block, no commands.
CPU is fine (~15-25 min, mostly download).

  from data_prep import prepare_data
  prepare_data()

Idempotent: if data_meta/index.json already exists on the Hub, it skips straight to printing
the existing split summary instead of re-downloading and rebuilding everything. Pass force=True
to rebuild anyway.

  1. Downloads the 16k-pair SEN1-2 subset from Kaggle (requiemonk) with kagglehub.
  2. Verifies every S1 file has an S2 partner and that images open / are 256x256.
  3. Builds TWO splits and pushes them to the Hub:
       split_scene.json  – leakage-free: no source scene in two splits (USE THIS)
       split_patch.json  – the published per-patch split (yuuIind/SAR2Optical) —
                           kept ONLY to measure how much per-patch leakage inflates scores
  4. Computes per-image mean RGB of the optical target and the leave-one-out
     scene-mean colour (used by the `sceneref` experiment).
  5. Prints a leakage audit.
"""
import json, urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image

from common import (DATA_META, KAGGLE_HANDLE, PATCH_SPLIT_URL, CLASSES, parse_s1, s2_of,
                    scene_split, log, preflight, sync_data_meta, hf_push_file, require_hf_token,
                    HF_RUNS_REPO)


def prepare_data(force=False):
    preflight()
    require_hf_token()

    if not force:
        got = sync_data_meta()
        if "index.json" in got and "split_scene.json" in got:
            log("prepare_data: index.json + split_scene.json already exist on the Hub -- SKIPPING "
                "(pass force=True to rebuild). Sanity: printing the existing split sizes.")
            root, index = _load_existing_summary()
            return {"skipped": True, "root": root, "n_items": len(index)}

    DATA_META.mkdir(parents=True, exist_ok=True)
    import kagglehub
    root = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
    size_gb = sum(f.stat().st_size for f in root.rglob("*") if f.is_file()) / 1e9
    log(f"dataset at {root}  ({size_gb:.2f} GB on disk)")

    base = None
    for cand in [root] + [p for p in root.rglob("*") if p.is_dir()]:
        if all((cand / c / "s1").is_dir() for c in CLASSES):
            base = cand; break
    assert base is not None, f"Could not find <class>/s1 folders under {root}; inspect the tree manually."
    log(f"class root: {base}")

    items, bad = [], []
    for c in CLASSES:
        for f in sorted((base / c / "s1").glob("*.png")):
            rel = f.relative_to(base).as_posix()
            meta = parse_s1(rel)
            s2 = s2_of(rel)
            if meta is None or not (base / s2).exists():
                bad.append(rel); continue
            items.append({**meta, "s1": rel, "s2": s2})
    log(f"pairs found: {len(items):,}   unmatched/unparsable: {len(bad)}")
    assert len(items) > 1000, f"only found {len(items)} pairs -- something is wrong with the download/layout"

    mean_rgb, broken = {}, []
    for k, it in enumerate(items):
        try:
            a = np.asarray(Image.open(base / it["s2"]).convert("RGB"), np.float32) / 255.
            s = Image.open(base / it["s1"]).size
            if a.shape[:2] != (256, 256) or s != (256, 256):
                broken.append(it["id"]); continue
            mean_rgb[it["id"]] = a.reshape(-1, 3).mean(0).tolist()
        except Exception:
            broken.append(it["id"])
        if k % 4000 == 0:
            log(f"  checked {k:,}/{len(items):,}")
    items = [it for it in items if it["id"] not in set(broken)]
    log(f"usable pairs: {len(items):,}   broken: {len(broken)}")
    assert len(items) > 1000, "too few usable pairs survived integrity checking"

    ssum, scount = defaultdict(lambda: np.zeros(3)), Counter()
    for it in items:
        ssum[it["scene"]] += mean_rgb[it["id"]]; scount[it["scene"]] += 1
    loo = {}
    for it in items:
        n = scount[it["scene"]]
        tot = ssum[it["scene"]]
        loo[it["id"]] = ((tot - np.array(mean_rgb[it["id"]])) / max(n - 1, 1)).tolist()

    sc_split, scene_assign = scene_split(items)
    assert all(len(sc_split[k]) > 0 for k in ("train", "val", "test")), "an empty split part -- bad scene_split()"
    try:
        pub = json.load(urllib.request.urlopen(PATCH_SPLIT_URL, timeout=30))["data"]
        id_of = {it["s1"]: it["id"] for it in items}
        patch_split = {k: [id_of[p] for p in v if p in id_of] for k, v in pub.items()}
    except Exception as e:
        log(f"could not fetch published split ({e}); patch split will be a seeded random split")
        rng = np.random.default_rng(42); ids = [it["id"] for it in items]; rng.shuffle(ids)
        n = len(ids); patch_split = {"train": ids[:int(.8 * n)], "val": ids[int(.8 * n):int(.9 * n)],
                                     "test": ids[int(.9 * n):]}

    test_scenes = {s for s, v in scene_assign.items() if v == "test"}
    by_id = {it["id"]: it for it in items}
    leak_eval = [i for i in patch_split["test"] if by_id[i]["scene"] in test_scenes]

    files = {
        "index.json": {"root": str(base), "items": items},
        "split_scene.json": sc_split,
        "split_patch.json": patch_split,
        "split_leak_eval.json": {"leak_eval": leak_eval},
        "color_stats.json": {"mean_rgb": mean_rgb, "scene_loo_rgb": loo, "scene_assign": scene_assign},
    }
    log("writing + pushing data_meta (this must survive session end)...")
    pushed = []
    for name, obj in files.items():
        p = DATA_META / name
        json.dump(obj, open(p, "w"))
        hf_push_file(p, f"data_meta/{name}")
        pushed.append(name)
    assert set(pushed) == set(files), f"not all files pushed: {set(files) - set(pushed)}"

    _print_audit(sc_split, patch_split, by_id, leak_eval, test_scenes, items)
    log(f"saved + pushed metadata -> {DATA_META}  and  hf://{HF_RUNS_REPO}/data_meta/")
    log("SANITY: all 5 files above showed a 'pushed ->' line, and the assert above confirmed it. "
        "If this cell completed without an AssertionError, data_meta is safely on the Hub.")
    return {"skipped": False, "root": str(base), "n_items": len(items)}


def _load_existing_summary():
    from common import load_index, load_split
    root, index = load_index()
    sp = load_split("scene")
    by_id = index
    for k, v in sp.items():
        cls = dict(Counter(by_id[i]["cls"] for i in v))
        log(f"   existing {k:5s}: {len(v)} images, classes {cls}")
    return str(root), index


def _print_audit(sc_split, patch_split, by_id, leak_eval, test_scenes, items):
    log("=" * 70)
    for name, sp in (("SCENE split", sc_split), ("PATCH split", patch_split)):
        scenes = {k: {by_id[i]["scene"] for i in v} for k, v in sp.items()}
        cls = {k: dict(Counter(by_id[i]["cls"] for i in v)) for k, v in sp.items()}
        log(f"{name}: sizes {({k: len(v) for k, v in sp.items()})}")
        log(f"   scenes per split {({k: len(v) for k, v in scenes.items()})}  "
            f"train\u2229test scenes = {len(scenes['train'] & scenes['test'])}")
        for k in sp: log(f"   {k:5s} classes {cls[k]}")
    log(f"leak-eval subset: {len(leak_eval)} images from {len(test_scenes)} held-out scenes")
    log(f"seasons: {dict(Counter(it['season'] for it in items))}")


Writing data_prep.py


In [4]:
from data_prep import prepare_data
prepare_data()


[15:13:33] ======================================================================
[15:13:33] PREFLIGHT  cache=/content/sar_fyp_cache  hf_repo=AliMusaRizvi/sar-to-optical-diffusion  token_set=True
[15:13:33]   GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
[15:13:33] ======================================================================


index.json:   0%|          | 0.00/3.17M [00:00<?, ?B/s]

split_scene.json:   0%|          | 0.00/407k [00:00<?, ?B/s]

split_patch.json:   0%|          | 0.00/407k [00:00<?, ?B/s]

split_leak_eval.json:   0%|          | 0.00/3.94k [00:00<?, ?B/s]

split_scene_curated.json:   0%|          | 0.00/397k [00:00<?, ?B/s]

... (4 outputs left out)

100%|██████████| 2.55G/2.55G [00:36<00:00, 74.4MB/s]

Extracting files...


[15:14:33] dataset root from a past session (/kaggle/input/sentinel12-image-pairs-segregated-by-terrain/v_2) is gone; re-downloaded to /root/.cache/kagglehub/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/versions/1
[15:14:33]    existing train: 12500 images, classes {'agri': 3056, 'barrenland': 2910, 'grassland': 3551, 'urban': 2983}
[15:14:33]    existing val  : 1843 images, classes {'agri': 527, 'barrenland': 456, 'grassland': 342, 'urban': 518}
[15:14:33]    existing test : 1657 images, classes {'agri': 416, 'grassland': 108, 'barrenland': 634, 'urban': 499}


{'skipped': True,
 'root': '/root/.cache/kagglehub/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/versions/1/v_2',
 'n_items': 15999}

## Step 1 -- oracle prechecks + `latent_scale.json` (idempotent: skips if already done)

In [5]:
%%writefile precheck.py
"""
precheck.py — call run_precheck() directly. No argparse, no commands. GPU, ~30-45 min on A100.

  from precheck import run_precheck
  run_precheck()

Idempotent: if data_meta/latent_scale.json already exists on the Hub, prints the existing GO/NO-GO
verdicts instead of re-running (pass force=True to redo). Must run AFTER prepare_data().

  A. Pipeline sanity on your existing regressor_v3_final (CONTAMINATED -- not a result, just a
     loading/normalisation check).
  B. Colour/illumination oracles: per-image affine ceiling vs. scene-reference colour shift.
  C. Frequency share in the Haar LL2 band -> justifies/kills `wavelet`.
  D. Dual-SAR latent ridge probe -> justifies/kills `dualsar`.
Also writes data_meta/latent_scale.json, which every train_one() call needs.
"""
import json
import numpy as np
import torch
import torch.nn.functional as F

from common import (DATA_META, OLD_HF_REPO, TOTAL_CH, PairDS, DataLoader, load_index, load_split,
                    load_vae, build_unet, build_text_embeds, get_cond, psnr, haar, _gauss_blur,
                    log, seed_all, preflight, sync_data_meta, hf_push_file, require_hf_token, dev)

N_ORACLE, N_PROBE_TR, N_PROBE_VA = 400, 300, 100


def _load_old_v3():
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    cfg = json.load(open(hf_hub_download(OLD_HF_REPO, "regressor_v3_final/bridge_config.json")))
    unet = build_unet(init_sd=False)
    unet.load_state_dict(load_file(hf_hub_download(OLD_HF_REPO, "regressor_v3_final/unet.safetensors")))
    try:
        bcfg = json.load(open(hf_hub_download(OLD_HF_REPO, "bridge_final/bridge_config.json")))
    except Exception:
        bcfg = {}
    return unet.to(dev).eval().float().requires_grad_(False), cfg, bcfg


def _affine_fit(p, g):
    out = torch.empty_like(p)
    for c in range(3):
        x, y = p[c].flatten(), g[c].flatten()
        A = torch.stack([x, torch.ones_like(x)], 1)
        sol = torch.linalg.lstsq(A, y[:, None]).solution
        out[c] = (A @ sol).view_as(p[c])
    return out


def _print_verdict(out):
    g = out["gains_db"]; probe = out["latent_ridge_R2"]
    log("=" * 72)
    log(f"A. sanity (CONTAMINATED, not a result): v3 PSNR on held-out scenes = {out['psnr']['base']:.2f} dB")
    log(f"B. per-image affine oracle        +{g['affine']:.2f} dB  (ceiling)")
    log(f"   scene-reference colour shift   +{g['scene_shift']:.2f} dB  "
        f"= {100 * g['scene_shift'] / max(g['affine'], 1e-6):.0f}% of the affine ceiling")
    log(f"   GT low-freq paste (\u03c3=2.5)      +{g['lowpass']:.2f} dB | GT high-freq paste +{g['highpass']:.2f} dB")
    log(f"C. share of squared error in Haar LL2 band: {100 * out['haar_LL2_error_share']:.1f}%")
    log(f"D. latent ridge R\u00b2  gray3={probe['gray3']:.4f}  dual={probe['dual']:.4f}")
    log("-" * 72)
    log("GO / NO-GO")
    log(f"  sceneref : {'GO' if g['scene_shift'] > 0.5 else 'WEAK'}  (GO if scene shift > +0.5 dB)")
    log(f"  wavelet  : {'GO' if out['haar_LL2_error_share'] > 0.25 else 'WEAK'}  (GO if LL2 share > 25%)")
    log(f"  dualsar  : {'GO' if probe['dual'] > probe['gray3'] + 0.005 else 'WEAK'}  (GO if R\u00b2 gain > 0.005)")


def run_precheck(force=False):
    preflight()
    require_hf_token()
    sync_data_meta()

    ls_path = DATA_META / "latent_scale.json"
    pr_path = DATA_META / "precheck_results.json"
    if not force and ls_path.exists() and pr_path.exists():
        log("run_precheck: latent_scale.json + precheck_results.json already on the Hub -- SKIPPING "
            "(pass force=True to redo). Printing the existing verdicts.")
        _print_verdict(json.load(open(pr_path)))
        return json.load(open(pr_path))

    assert (DATA_META / "index.json").exists(), "run prepare_data() first (data_prep.py)"
    seed_all()
    root, index = load_index()
    sp = load_split("scene")
    colors = json.load(open(DATA_META / "color_stats.json"))
    vae = load_vae(dev)
    v3, rcfg, bcfg = _load_old_v3()
    scale = float(rcfg.get("latent_scale", 1.0)); tb = int(rcfg.get("fixed_tb", 933))
    json.dump({"latent_scale": scale, "fixed_tb": tb, "bridge": {k: bcfg.get(k) for k in
              ("T", "kappa", "eta_1", "eta_T", "latent_scale")}}, open(ls_path, "w"), indent=2)
    hf_push_file(ls_path, "data_meta/latent_scale.json")
    log(f"latent_scale={scale}  fixed_tb={tb}  (pushed -- train_one() needs this)")
    emb = build_text_embeds(dev)

    rng = np.random.default_rng(0)
    test_ids = list(rng.choice(sp["test"], min(N_ORACLE, len(sp["test"])), replace=False))
    dl = DataLoader(PairDS(test_ids, root, index), batch_size=8, num_workers=4)
    R = {k: [] for k in ("base", "affine", "scene_shift", "lowpass", "highpass")}
    ll_share = []
    for b in dl:
        sar, opt = b["sar"].to(dev), b["opt"].to(dev)
        with torch.no_grad():
            y = vae.enc(sar) * scale
            c = get_cond(b["season"], b["cls"], emb).to(dev)
            tbv = torch.full((y.shape[0],), tb, device=dev, dtype=torch.long)
            pred = vae.dec(v3(torch.cat([y, y], 1), tbv, encoder_hidden_states=c).sample / scale).clamp(-1, 1)
        for j in range(pred.shape[0]):
            p, g = pred[j], opt[j]
            R["base"].append(psnr(p, g))
            R["affine"].append(psnr(_affine_fit(p, g).clamp(-1, 1), g))
            ref = torch.tensor(colors["scene_loo_rgb"][b["id"][j]], device=dev) * 2 - 1
            shifted = p - p.mean((1, 2), keepdim=True) + ref[:, None, None]
            R["scene_shift"].append(psnr(shifted.clamp(-1, 1), g))
            lp_g, lp_p = _gauss_blur(g[None], 2.5)[0], _gauss_blur(p[None], 2.5)[0]
            R["lowpass"].append(psnr((p - lp_p + lp_g).clamp(-1, 1), g))
            R["highpass"].append(psnr((lp_p + (g - lp_g)).clamp(-1, 1), g))
            e = (p - g)[None]
            ll, *_ = haar(e); ll2, *_ = haar(ll)
            ll_share.append(float((ll2 ** 2).sum() * 1.0 / (e ** 2).sum().clamp_min(1e-12)))
    M = {k: float(np.mean(v)) for k, v in R.items()}
    assert all(np.isfinite(v) for v in M.values()), f"non-finite oracle metric: {M}"

    def latents(ids, mode):
        Zs, Zo = [], []
        for b in DataLoader(PairDS(ids, root, index, sar_mode=mode), batch_size=16, num_workers=4):
            with torch.no_grad():
                Zs.append(vae.enc(b["sar"].to(dev)).float()); Zo.append(vae.enc(b["opt"].to(dev)).float())
        return torch.cat(Zs), torch.cat(Zo)

    def feats(z):
        u = F.unfold(z, 3, padding=1)
        return u.transpose(1, 2).reshape(-1, u.shape[1])

    tr_ids = list(rng.choice(sp["train"], N_PROBE_TR, replace=False))
    va_ids = list(rng.choice(sp["val"], N_PROBE_VA, replace=False))
    probe = {}
    for mode in ("gray3", "dual"):
        zs_tr, zo_tr = latents(tr_ids, mode); zs_va, zo_va = latents(va_ids, mode)
        X, Y = feats(zs_tr), zo_tr.permute(0, 2, 3, 1).reshape(-1, TOTAL_CH)
        Xv, Yv = feats(zs_va), zo_va.permute(0, 2, 3, 1).reshape(-1, TOTAL_CH)
        mu, sd = X.mean(0), X.std(0) + 1e-6
        X, Xv = (X - mu) / sd, (Xv - mu) / sd
        X1 = torch.cat([X, torch.ones(len(X), 1, device=dev)], 1)
        Xv1 = torch.cat([Xv, torch.ones(len(Xv), 1, device=dev)], 1)
        W = torch.linalg.solve(X1.T @ X1 + 10.0 * torch.eye(X1.shape[1], device=dev), X1.T @ Y)
        res = ((Xv1 @ W - Yv) ** 2).sum(0); tot = ((Yv - Yv.mean(0)) ** 2).sum(0)
        probe[mode] = float((1 - res / tot).mean())

    out = {"n_images": len(R["base"]), "psnr": M,
           "gains_db": {k: M[k] - M["base"] for k in M if k != "base"},
           "haar_LL2_error_share": float(np.mean(ll_share)),
           "latent_ridge_R2": probe}
    json.dump(out, open(pr_path, "w"), indent=2)
    hf_push_file(pr_path, "data_meta/precheck_results.json")

    _print_verdict(out)
    log(f"saved + pushed -> {pr_path}")
    return out


Writing precheck.py


In [6]:
from precheck import run_precheck
run_precheck()


[15:14:33] ======================================================================
[15:14:33] PREFLIGHT  cache=/content/sar_fyp_cache  hf_repo=AliMusaRizvi/sar-to-optical-diffusion  token_set=True
[15:14:33]   GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
[15:14:33] ======================================================================
[15:14:33] data_meta synced from hf://AliMusaRizvi/sar-to-optical-diffusion: ['index.json', 'split_scene.json', 'split_patch.json', 'split_leak_eval.json', 'split_scene_curated.json', 'color_stats.json', 'latent_scale.json', 'precheck_results.json']
[15:14:33] run_precheck: latent_scale.json + precheck_results.json already on the Hub -- SKIPPING (pass force=True to redo). Printing the existing verdicts.
[15:14:33] ========================================================================
[15:14:33] A. sanity (CONTAMINATED, not a result): v3 PSNR on held-out scenes = 15.05 dB
[15:14:33] B. per-image affine oracle        +1.46 dB  (ceiling)
[15:14:33]    scene-refere

{'n_images': 400,
 'psnr': {'base': 15.054311211353484,
  'affine': 16.513784608103933,
  'scene_shift': 14.987643421974404,
  'lowpass': 22.921901486308695,
  'highpass': 16.88908174681961},
 'gains_db': {'affine': 1.4594733967504485,
  'scene_shift': -0.06666778937908013,
  'lowpass': 7.8675902749552105,
  'highpass': 1.8347705354661255},
 'haar_LL2_error_share': 0.7995676720142364,
 'latent_ridge_R2': {'gray3': 0.09785884618759155,
  'dual': 0.10817985981702805}}

## Sanity check -- the 6 core experiments (run this before any training cell below)

5 real steps of each, no Hub push, ~2-5 minutes total. `curated` is NOT included here -- it needs
`curate_data()` (further down) to have run first, so it gets its own check right after that cell
instead of failing here on a prerequisite that doesn't exist yet.

**If anything shows FAIL, stop and fix it before continuing.**


In [7]:
from common import sanity_check_all, DEFAULT_EXPS
_sanity_results = sanity_check_all(exps=list(DEFAULT_EXPS))
assert all(r["ok"] for r in _sanity_results.values()), "Fix the FAILs above before training for real."


[15:14:33] ======================================================================
[15:14:33] SANITY CHECK: validating ['baseline', 'patchsplit', 'dualsar', 'wavelet', 'sceneref', 'bridge'] end-to-end (5 steps each, no Hub push)
[15:14:33] ======================================================================
[15:14:33]   checking [baseline] ...
[15:14:38] dataset root from a past session (/kaggle/input/sentinel12-image-pairs-segregated-by-terrain/v_2) is gone; re-downloaded to /root/.cache/kagglehub/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/versions/1


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

vae/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

adapted16_final/base_vae.pth: reconstructing file:   0%|          |  0.00B /  335MB            

... (22 outputs left out)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[15:16:36]   [sceneref] PASS  losses=[2.3466, 2.0746, 1.9812, 2.0917, 2.0204]  ckpt_roundtrip=True
[15:16:36]   checking [bridge] ...
Using Colab cache for faster access to the 'sentinel12-image-pairs-segregated-by-terrain' dataset.
[15:16:39] dataset root from a past session (/kaggle/input/sentinel12-image-pairs-segregated-by-terrain/v_2) is gone; re-downloaded to /kaggle/input/sentinel12-image-pairs-segregated-by-terrain


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[15:17:37]   [bridge] PASS  losses=[1.332, 1.1769, 1.0821, 1.107, 1.1472]  ckpt_roundtrip=True
[15:17:37] ======================================================================
[15:17:37] ALL SANITY CHECKS PASSED. Safe to run the training cells below.


### `baseline`
Reference row: per-scene split, [raw,raw,raw] SAR, standard loss.

**Resumable, and safe to re-run:** if this cell stops for any reason, just re-run this exact cell
(after redoing setup + `%%writefile common.py` if the runtime restarted) -- it picks up from its
last checkpoint automatically. If it's already finished, re-running this cell costs about a
second (checked before any download happens).


In [8]:
from common import train_one
train_one("baseline")


state.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

[15:17:39] [baseline] already finished (step 15000 >= steps=15000) -- skipping immediately, nothing downloaded or loaded.


'already_done'

### `patchsplit`
Same as baseline but trained on the published per-PATCH split -- quantifies leakage size (compare against baseline via evaluate_experiments(leak_audit=True) later).

**Resumable, and safe to re-run:** if this cell stops for any reason, just re-run this exact cell
(after redoing setup + `%%writefile common.py` if the runtime restarted) -- it picks up from its
last checkpoint automatically. If it's already finished, re-running this cell costs about a
second (checked before any download happens).


In [9]:
from common import train_one
train_one("patchsplit")


state.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

[15:17:39] [patchsplit] already finished (step 15000 >= steps=15000) -- skipping immediately, nothing downloaded or loaded.


'already_done'

### `dualsar`
SAR input becomes [raw, multilook5, multilook11] -- adds a lower-variance backscatter estimate without removing the speckle-bearing raw channel.

**Resumable, and safe to re-run:** if this cell stops for any reason, just re-run this exact cell
(after redoing setup + `%%writefile common.py` if the runtime restarted) -- it picks up from its
last checkpoint automatically. If it's already finished, re-running this cell costs about a
second (checked before any download happens).


In [10]:
from common import train_one
train_one("dualsar")


state.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

[15:17:39] [dualsar] already finished (step 15000 >= steps=15000) -- skipping immediately, nothing downloaded or loaded.


'already_done'

### `wavelet`
Image L1 replaced by the (now correctly scale-normalized) 2-level Haar wavelet L1, double-weighted on the low-frequency band.

**Resumable, and safe to re-run:** if this cell stops for any reason, just re-run this exact cell
(after redoing setup + `%%writefile common.py` if the runtime restarted) -- it picks up from its
last checkpoint automatically. If it's already finished, re-running this cell costs about a
second (checked before any download happens).


In [11]:
from common import train_one
train_one("wavelet")


state.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

[15:17:39] [wavelet] already finished (step 15000 >= steps=15000) -- skipping immediately, nothing downloaded or loaded.


'already_done'

### `sceneref`
Second UNet input is the scene's leave-one-out reference colour instead of a copy of z_sar -- an upper bound on what an external colour reference could buy.

**Resumable, and safe to re-run:** if this cell stops for any reason, just re-run this exact cell
(after redoing setup + `%%writefile common.py` if the runtime restarted) -- it picks up from its
last checkpoint automatically. If it's already finished, re-running this cell costs about a
second (checked before any download happens).


In [12]:
from common import train_one
train_one("sceneref")


state.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

[15:17:39] [sceneref] already finished (step 15000 >= steps=15000) -- skipping immediately, nothing downloaded or loaded.


'already_done'

### `bridge`
The ResShift stochastic branch (T=15, kappa=2), retrained here for a same-setting perception-distortion comparison.

**Resumable, and safe to re-run:** if this cell stops for any reason, just re-run this exact cell
(after redoing setup + `%%writefile common.py` if the runtime restarted) -- it picks up from its
last checkpoint automatically. If it's already finished, re-running this cell costs about a
second (checked before any download happens).


In [13]:
from common import train_one
train_one("bridge")


state.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

[15:17:40] [bridge] already finished (step 15000 >= steps=15000) -- skipping immediately, nothing downloaded or loaded.


'already_done'

## Data curation + the `curated` experiment
Scores every training pair under `baseline`, flags degenerate/hardest-to-predict outliers, and
writes a filtered split. Needs `baseline` (above) already trained. Idempotent like the steps above.


In [14]:
%%writefile curate.py
"""
curate.py — call curate_data() directly. No argparse, no commands. Needs 'baseline' trained.

  from curate import curate_data
  curate_data()
  # then just:
  from common import train_one
  train_one("curated")

Two independent flags, both computed on the TRAIN split only (val/test untouched, so comparing
to 'baseline' stays apples-to-apples on identical eval data):

  1. Degenerate images: near-zero variance in the optical target (flat cloud fill, no-data,
     snow-out) or the SAR input (dead sensor patch) -- cheap, no model needed.
  2. Hardest-to-predict under baseline: top drop_pct by (1 - SSIM) against baseline's own
     prediction -- a proxy for cloud contamination / heavy misregistration / genuine SAR-optical
     disagreement. Defaults to a conservative 3%. Set drop_pct=0 to use flag 1 only.
"""
import json
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from common import (CACHE, RUNS, PairDS, DataLoader, load_index, load_split, load_run,
                    build_text_embeds, get_cond, log, seed_all, preflight, sync_data_meta,
                    require_hf_token, hf_push_file, dev)


@torch.no_grad()
def curate_data(drop_pct=3.0, std_thresh=0.02, force=False):
    preflight(); require_hf_token()

    if not force:
        got = sync_data_meta()
        if "split_scene_curated.json" in got:
            log("curate_data: split_scene_curated.json already exists on the Hub -- SKIPPING "
                "(pass force=True to redo). Printing its existing summary.")
            curated = json.load(open(CACHE / "data_meta" / "split_scene_curated.json"))
            log(f"   existing curated split: train={len(curated['train'])} "
                f"val={len(curated['val'])} test={len(curated['test'])}")
            return curated

    sync_data_meta(); seed_all()
    root, index = load_index()
    sp = load_split("scene")
    train_ids = sp["train"]
    assert len(train_ids) > 0, "empty train split -- run prepare_data() first"

    degenerate = []
    for i in train_ids:
        it = index[i]
        opt = np.asarray(Image.open(Path(root) / it["s2"]).convert("RGB"), np.float32) / 255.
        sar = np.asarray(Image.open(Path(root) / it["s1"]).convert("L"), np.float32) / 255.
        if opt.std() < std_thresh or sar.std() < std_thresh:
            degenerate.append(i)
    log(f"flag 1 (degenerate): {len(degenerate)}/{len(train_ids)} "
        f"({100 * len(degenerate) / len(train_ids):.2f}%)")

    hardest = []
    if drop_pct > 0:
        from torchmetrics.functional.image import structural_similarity_index_measure as ssim_f
        unet, vae, cfg = load_run(RUNS / "baseline", dev, tag="best", exp="baseline")
        emb = build_text_embeds(dev)
        scale, tb = float(cfg["latent_scale"]), int(cfg["fixed_tb"])
        dl = DataLoader(PairDS(train_ids, root, index, sar_mode="gray3"), batch_size=16, num_workers=4)
        scored = []
        for b in dl:
            sar, opt = b["sar"].to(dev), b["opt"].to(dev)
            y = vae.enc(sar) * scale
            c = get_cond(b["season"], b["cls"], emb).to(dev)
            tbv = torch.full((y.shape[0],), tb, device=dev, dtype=torch.long)
            rec = vae.dec(unet(torch.cat([y, y], 1), tbv, encoder_hidden_states=c).sample / scale).clamp(-1, 1)
            for j in range(rec.shape[0]):
                s = float(ssim_f(rec[j:j + 1], opt[j:j + 1], data_range=2.0))
                scored.append((b["id"][j], s))
        scored.sort(key=lambda x: x[1])
        n_drop = int(len(scored) * drop_pct / 100)
        hardest = [i for i, s in scored[:n_drop]]
        log(f"flag 2 (hardest {drop_pct}%): {len(hardest)} images, "
            f"SSIM range dropped = [{scored[0][1]:.3f}, {scored[n_drop - 1][1]:.3f}]")
        del unet, vae; torch.cuda.empty_cache()

    dropped = set(degenerate) | set(hardest)
    curated_train = [i for i in train_ids if i not in dropped]
    assert len(curated_train) > 0.5 * len(train_ids), \
        f"curation dropped over half the train set ({len(dropped)}/{len(train_ids)}) -- thresholds too aggressive"
    log(f"TOTAL dropped: {len(dropped)}/{len(train_ids)} ({100 * len(dropped) / len(train_ids):.2f}%) "
        f"-> curated train = {len(curated_train)}")

    curated = {"train": curated_train, "val": sp["val"], "test": sp["test"]}
    p = CACHE / "data_meta" / "split_scene_curated.json"; p.parent.mkdir(parents=True, exist_ok=True)
    json.dump(curated, open(p, "w"))
    json.dump({"degenerate": degenerate, "hardest": hardest, "drop_pct": drop_pct, "std_thresh": std_thresh},
              open(CACHE / "data_meta" / "curation_audit.json", "w"))
    hf_push_file(p, "data_meta/split_scene_curated.json")
    hf_push_file(CACHE / "data_meta" / "curation_audit.json", "data_meta/curation_audit.json")
    log("Next: from common import train_one; train_one('curated')")
    return curated


Writing curate.py


In [15]:
from curate import curate_data
curate_data()


[15:17:40] ======================================================================
[15:17:40] PREFLIGHT  cache=/content/sar_fyp_cache  hf_repo=AliMusaRizvi/sar-to-optical-diffusion  token_set=True
[15:17:40]   GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
[15:17:40] ======================================================================
[15:17:40] data_meta synced from hf://AliMusaRizvi/sar-to-optical-diffusion: ['index.json', 'split_scene.json', 'split_patch.json', 'split_leak_eval.json', 'split_scene_curated.json', 'color_stats.json', 'latent_scale.json', 'precheck_results.json']
[15:17:40] curate_data: split_scene_curated.json already exists on the Hub -- SKIPPING (pass force=True to redo). Printing its existing summary.
[15:17:40]    existing curated split: train=12125 val=1843 test=1657


{'train': ['ROIs1868_summer_59_p10',
  'ROIs1868_summer_59_p100',
  'ROIs1868_summer_59_p1000',
  'ROIs1868_summer_59_p1001',
  'ROIs1868_summer_59_p1002',
  'ROIs1868_summer_59_p1003',
  'ROIs1868_summer_59_p1004',
  'ROIs1868_summer_59_p1005',
  'ROIs1868_summer_59_p1006',
  'ROIs1868_summer_59_p1007',
  'ROIs1868_summer_59_p1008',
  'ROIs1868_summer_59_p1009',
  'ROIs1868_summer_59_p101',
  'ROIs1868_summer_59_p1010',
  'ROIs1868_summer_59_p1011',
  'ROIs1868_summer_59_p1012',
  'ROIs1868_summer_59_p1013',
  'ROIs1868_summer_59_p1014',
  'ROIs1868_summer_59_p1015',
  'ROIs1868_summer_59_p1016',
  'ROIs1868_summer_59_p1017',
  'ROIs1868_summer_59_p1018',
  'ROIs1868_summer_59_p1019',
  'ROIs1868_summer_59_p102',
  'ROIs1868_summer_59_p1020',
  'ROIs1868_summer_59_p1021',
  'ROIs1868_summer_59_p1022',
  'ROIs1868_summer_59_p1023',
  'ROIs1868_summer_59_p1024',
  'ROIs1868_summer_59_p1025',
  'ROIs1868_summer_59_p1026',
  'ROIs1868_summer_59_p1027',
  'ROIs1868_summer_59_p1028',
  'ROI

### `curated`'s own sanity check
Runs now, not earlier, because `split_scene_curated.json` only exists after the cell above.


In [16]:
from common import sanity_check
_r = sanity_check("curated")
print("curated sanity check:", "PASS" if _r["ok"] else "FAIL", _r["losses"], _r.get("error"))
assert _r["ok"], f"curated sanity check failed: {_r['error']}"


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

curated sanity check: PASS [2.4481, 2.1581, 2.0654, 2.1155, 2.0292] None


### `curated`
Trained on the outlier-filtered split from the cells above, same architecture/losses as `baseline`.

**Resumable, and safe to re-run:** same as every experiment above.


In [17]:
from common import train_one
train_one("curated")


state.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

[15:17:53] [curated] cfg={'split': 'scene_curated', 'sar_mode': 'gray3', 'cond': 'self', 'loss': 'std', 'model': 'regressor'}  train=12,125 val=1,843


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

fyp_runs/curated/last/unet.safetensors: reconstructing file:   0%|          |  0.00B / 1.72GB            

fyp_runs/curated/last/unet.safetensors: downloading bytes:           |  0.00B            

fyp_runs/curated/last/decoder.pth: reconstructing file:   0%|          |  0.00B /  198MB            

fyp_runs/curated/last/decoder.pth: downloading bytes:           |  0.00B            

[15:18:11] [curated] RESUMING from step 9000 (best val PSNR so far 14.820)
[15:18:12] [curated] >>> TRAINING LOOP ENTERED, step 9000 -> 15000 <<<


/content/sar_fyp/common.py:810: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  parts = {k: float(v) for k, v in parts.items()}


[15:19:16] [curated] step 9100/15000 loss 1.3413 lat_shift=0.468 lat_l1=0.469 img_l1=0.226 ssim=0.736 mind=0.165 sam=0.119 | 0.02 h
[15:20:18] [curated] step 9200/15000 loss 1.3681 lat_shift=0.454 lat_l1=0.455 img_l1=0.240 ssim=0.812 mind=0.151 sam=0.108 | 0.04 h
[15:21:20] [curated] step 9300/15000 loss 1.3151 lat_shift=0.421 lat_l1=0.421 img_l1=0.233 ssim=0.821 mind=0.150 sam=0.105 | 0.06 h
[15:22:22] [curated] step 9400/15000 loss 1.3075 lat_shift=0.417 lat_l1=0.416 img_l1=0.230 ssim=0.827 mind=0.152 sam=0.089 | 0.07 h
[15:23:24] [curated] step 9500/15000 loss 1.1985 lat_shift=0.389 lat_l1=0.391 img_l1=0.185 ssim=0.774 mind=0.151 sam=0.109 | 0.09 h
[15:23:24] [curated]   mem: RAM(peak-so-far) 9.55 GB  VRAM(alloc) 14.81 GB  VRAM(reserved) 23.92 GB
[15:24:27] [curated] step 9600/15000 loss 1.2818 lat_shift=0.409 lat_l1=0.411 img_l1=0.228 ssim=0.799 mind=0.150 sam=0.101 | 0.11 h
[15:25:29] [curated] step 9700/15000 loss 1.2606 lat_shift=0.406 lat_l1=0.405 img_l1=0.220 ssim=0.785 mind=0

'done'

## Final comparison table
Scene-bootstrap 95% CIs, paired delta vs `baseline`, and a leakage audit. Run any time after the
checkpoints above exist -- this cell doesn't need to run in the same session that trained them,
and it always re-evaluates in full by design (so its numbers reflect whatever currently exists).


In [18]:
%%writefile evaluate.py
"""
evaluate.py — call evaluate_experiments() directly. No argparse, no commands.
~10-20 min per regressor, ~30-40 min for bridge (A100).

  from evaluate import evaluate_experiments
  evaluate_experiments(["baseline", "patchsplit", "dualsar", "wavelet", "sceneref", "bridge", "curated"])
  evaluate_experiments(leak_audit=True)   # leakage audit: same images, patch- vs scene-trained model

Outputs pushed to hf://<HF_RUNS_REPO>/eval_results/<tag>/ (cached locally at CACHE/eval/<tag>/):
  per_image_<exp>.csv, summary.json, table.md, comparison_grid.png, samples/{sar,gt,<exp>}/*.png
Statistics: mean +/- 95% CI from a SCENE-level bootstrap (patches from one scene are not
independent), and paired delta vs baseline with its own scene-bootstrap CI. A difference whose
CI crosses 0 is not a result -- say so in the report.
"""
import json, csv
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from common import (DATA_META, RUNS, CACHE, CLASSES, PairDS, DataLoader, load_index, load_split,
                    load_run, build_text_embeds, get_cond, resshift_eta, resshift_sample, psnr,
                    sam_deg, ergas, cc, to01, scene_bootstrap_ci, log, seed_all, preflight,
                    sync_data_meta, require_hf_token, hf_push_dir, HF_RUNS_REPO, dev)

METRICS = ("psnr", "ssim", "lpips", "sam", "ergas", "cc")


@torch.no_grad()
def _predict(unet, vae, cfg, b, emb, gen):
    scale, tb = float(cfg["latent_scale"]), int(cfg["fixed_tb"])
    sar = b["sar"].to(dev)
    y = vae.enc(sar) * scale
    c = get_cond(b["season"], b["cls"], emb).to(dev)
    if cfg["model"] == "bridge":
        eta = resshift_eta(cfg["T"], cfg["eta_1"], cfg["eta_T"]).to(dev)
        lat = resshift_sample(unet, y, c, eta, cfg["kappa"], cfg["T"], gen=gen)
    else:
        second = vae.enc(b["ref"].to(dev)) * scale if cfg["cond"] == "ref" else y
        tbv = torch.full((y.shape[0],), tb, device=dev, dtype=torch.long)
        lat = unet(torch.cat([y, second], 1), tbv, encoder_hidden_states=c).sample
    return vae.dec(lat / scale).clamp(-1, 1)


def _save_png(t, path):
    Image.fromarray((to01(t).permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)).save(path)


def _evaluate_one(exp, ids, root, index, emb, out_dir, save_ids, bs):
    import lpips as lpips_lib
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.functional.image import structural_similarity_index_measure as ssim_f
    unet, vae, cfg = load_run(RUNS / exp, dev, tag="best", exp=exp)
    ref = json.load(open(DATA_META / "color_stats.json"))["scene_loo_rgb"] if cfg["cond"] == "ref" else None
    ds = PairDS(ids, root, index, sar_mode=cfg["sar_mode"], ref_rgb=ref)
    dl = DataLoader(ds, batch_size=bs, num_workers=4)
    lp = lpips_lib.LPIPS(net="alex", verbose=False).to(dev)
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(dev)
    gen = torch.Generator(device=dev).manual_seed(0)
    rows = []
    (out_dir / "samples" / exp).mkdir(parents=True, exist_ok=True)
    for b in dl:
        rec = _predict(unet, vae, cfg, b, emb, gen)
        opt = b["opt"].to(dev)
        for j in range(rec.shape[0]):
            p, g = rec[j:j + 1], opt[j:j + 1]
            rows.append({"id": b["id"][j], "scene": b["scene"][j], "cls": b["cls"][j],
                         "psnr": psnr(p, g), "ssim": float(ssim_f(p, g, data_range=2.0)),
                         "lpips": float(lp(p, g)), "sam": sam_deg(p, g), "ergas": ergas(p, g), "cc": cc(p, g)})
            if b["id"][j] in save_ids:
                _save_png(rec[j], out_dir / "samples" / exp / f"{b['id'][j]}.png")
        fid.update(to01(opt), real=True); fid.update(to01(rec), real=False)
    assert rows, f"no images evaluated for {exp}"
    with open(out_dir / f"per_image_{exp}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    summ = {"fid": float(fid.compute()), "n": len(rows), "best_step": cfg.get("best_step")}
    sc = [r["scene"] for r in rows]
    for m in METRICS:
        v = [r[m] for r in rows]
        assert all(np.isfinite(v)), f"non-finite {m} for {exp}"
        summ[m] = float(np.mean(v)); summ[m + "_ci95"] = scene_bootstrap_ci(v, sc)
    summ["by_class"] = {c: {m: float(np.mean([r[m] for r in rows if r["cls"] == c])) for m in METRICS}
                        for c in CLASSES if any(r["cls"] == c for r in rows)}
    del unet, vae; torch.cuda.empty_cache()
    log(f"{exp:10s} PSNR {summ['psnr']:.3f} [{summ['psnr_ci95'][0]:.2f},{summ['psnr_ci95'][1]:.2f}] "
        f"SSIM {summ['ssim']:.4f} LPIPS {summ['lpips']:.4f} FID {summ['fid']:.1f} SAM {summ['sam']:.2f} "
        f"ERGAS {summ['ergas']:.2f} CC {summ['cc']:.3f}")
    return summ, rows


def _paired_delta(rows_a, rows_b, metric):
    a = {r["id"]: r for r in rows_a}
    d = [(r[metric] - a[r["id"]][metric], r["scene"]) for r in rows_b if r["id"] in a]
    vals, sc = [x for x, _ in d], [s for _, s in d]
    return float(np.mean(vals)), scene_bootstrap_ci(vals, sc)


def _grid(out_dir, exps, ids, root, index):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    cols = ["sar", "gt"] + exps
    fig, ax = plt.subplots(len(ids), len(cols), figsize=(2.4 * len(cols), 2.4 * len(ids)))
    for r, i in enumerate(ids):
        for c, name in enumerate(cols):
            p = out_dir / "samples" / name / f"{i}.png"
            ax[r, c].imshow(Image.open(p)) if p.exists() else None
            ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
            if r == 0: ax[r, c].set_title(name, fontsize=10)
        ax[r, 0].set_ylabel(index[i]["cls"], fontsize=10)
    plt.tight_layout(); plt.savefig(out_dir / "comparison_grid.png", dpi=130); plt.close()


def evaluate_experiments(exps=None, leak_audit=False, n_save_per_class=16, bs=8):
    preflight(); require_hf_token(); sync_data_meta(); seed_all()
    root, index = load_index()
    emb = build_text_embeds(dev)
    if leak_audit:
        tag, ids = "leak_audit", json.load(open(DATA_META / "split_leak_eval.json"))["leak_eval"]
        exps = ["baseline", "patchsplit"]
    else:
        exps = exps or ["baseline", "patchsplit", "dualsar", "wavelet", "sceneref", "bridge"]
        tag, ids = "scene_test", load_split("scene")["test"]
        if "patchsplit" in exps:
            log("WARNING: patchsplit trained on neighbours of these scenes -> its row is contaminated.")
    out_dir = CACHE / "eval" / tag; out_dir.mkdir(parents=True, exist_ok=True)
    exps = [e for e in exps if (RUNS / e / "best" / "config.json").exists()
           or log(f"skip {e}: no local checkpoint yet")]
    assert exps, "none of the requested experiments have a checkpoint yet"

    rng = np.random.default_rng(0); save_ids = set()
    for c in CLASSES:
        pool = [i for i in ids if index[i]["cls"] == c]
        save_ids |= set(rng.choice(pool, min(n_save_per_class, len(pool)), replace=False).tolist()) if pool else set()
    for sub in ("sar", "gt"): (out_dir / "samples" / sub).mkdir(parents=True, exist_ok=True)
    ds = PairDS(sorted(save_ids), root, index)
    for k in range(len(ds)):
        d = ds[k]; _save_png(d["sar"], out_dir / "samples" / "sar" / f"{d['id']}.png")
        _save_png(d["opt"], out_dir / "samples" / "gt" / f"{d['id']}.png")
    json.dump({i: {"cls": index[i]["cls"], "season": index[i]["season"], "scene": index[i]["scene"]}
               for i in save_ids}, open(out_dir / "samples" / "meta.json", "w"), indent=1)

    results, rows = {}, {}
    for e in exps:
        results[e], rows[e] = _evaluate_one(e, ids, root, index, emb, out_dir, save_ids, bs)
    if "baseline" in rows:
        for e in exps:
            if e == "baseline": continue
            results[e]["delta_vs_baseline"] = {m: _paired_delta(rows["baseline"], rows[e], m) for m in METRICS}
    json.dump(results, open(out_dir / "summary.json", "w"), indent=2)

    lines = ["| model | PSNR (95% CI) | \u0394PSNR vs baseline (95% CI) | SSIM | LPIPS | FID | SAM\u00b0 | ERGAS | CC |",
             "|---|---|---|---|---|---|---|---|---|"]
    for e, s in results.items():
        d = s.get("delta_vs_baseline", {}).get("psnr")
        dtxt = "\u2014" if d is None else f"{d[0]:+.2f} ({d[1][0]:+.2f}, {d[1][1]:+.2f})"
        lines.append(f"| {e} | {s['psnr']:.2f} ({s['psnr_ci95'][0]:.2f}\u2013{s['psnr_ci95'][1]:.2f}) | {dtxt} | "
                     f"{s['ssim']:.3f} | {s['lpips']:.3f} | {s['fid']:.1f} | {s['sam']:.2f} | {s['ergas']:.2f} | {s['cc']:.3f} |")
    (out_dir / "table.md").write_text("\n".join(lines) + "\n")
    print("\n".join(lines))
    if leak_audit and "patchsplit" in results:
        d = results["patchsplit"]["delta_vs_baseline"]["psnr"]
        log(f"LEAKAGE: per-patch split inflates PSNR by {d[0]:+.2f} dB (95% CI {d[1][0]:+.2f}..{d[1][1]:+.2f}) "
            f"on identical images.")
    pick = []
    for c in CLASSES:
        pool = sorted(i for i in save_ids if index[i]["cls"] == c)
        pick += pool[:1]
    _grid(out_dir, exps, pick, root, index)
    hf_push_dir(out_dir, f"eval_results/{tag}")
    log(f"saved locally -> {out_dir}  and pushed -> hf://{HF_RUNS_REPO}/eval_results/{tag}/")
    return results


Writing evaluate.py


In [19]:
from evaluate import evaluate_experiments
evaluate_experiments(["baseline", "patchsplit", "dualsar", "wavelet", "sceneref", "bridge", "curated"])


[16:23:53] ======================================================================
[16:23:53] PREFLIGHT  cache=/content/sar_fyp_cache  hf_repo=AliMusaRizvi/sar-to-optical-diffusion  token_set=True
[16:23:53]   GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
[16:23:53] ======================================================================
[16:23:53] data_meta synced from hf://AliMusaRizvi/sar-to-optical-diffusion: ['index.json', 'split_scene.json', 'split_patch.json', 'split_leak_eval.json', 'split_scene_curated.json', 'color_stats.json', 'latent_scale.json', 'precheck_results.json']


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[16:23:54] WARNING: patchsplit trained on neighbours of these scenes -> its row is contaminated.
[16:23:54] skip baseline: no local checkpoint yet
[16:23:54] skip patchsplit: no local checkpoint yet
[16:23:54] skip dualsar: no local checkpoint yet
[16:23:54] skip wavelet: no local checkpoint yet
[16:23:54] skip sceneref: no local checkpoint yet
[16:23:54] skip bridge: no local checkpoint yet


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 226MB/s]
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 182MB/s]


[16:25:07] curated    PSNR 13.127 [12.12,14.05] SSIM 0.1360 LPIPS 0.6943 FID 242.1 SAM 7.34 ERGAS 61.21 CC 0.176
| model | PSNR (95% CI) | ΔPSNR vs baseline (95% CI) | SSIM | LPIPS | FID | SAM° | ERGAS | CC |
|---|---|---|---|---|---|---|---|---|
| curated | 13.13 (12.12–14.05) | — | 0.136 | 0.694 | 242.1 | 7.34 | 61.21 | 0.176 |
[16:25:12]   pushed -> hf://AliMusaRizvi/sar-to-optical-diffusion/eval_results/scene_test/
[16:25:12] saved locally -> /content/sar_fyp_cache/eval/scene_test  and pushed -> hf://AliMusaRizvi/sar-to-optical-diffusion/eval_results/scene_test/


{'curated': {'fid': 242.08001708984375,
  'n': 1657,
  'best_step': 10000,
  'psnr': 13.126578953964307,
  'psnr_ci95': (12.122447305648201, 14.049503273114203),
  'ssim': 0.13604556410614524,
  'ssim_ci95': (0.09101659647156432, 0.19937457893602037),
  'lpips': 0.6943135125563673,
  'lpips_ci95': (0.6690291835095454, 0.7241335289719995),
  'sam': 7.33647993248345,
  'sam_ci95': (5.63140464252751, 9.902409572807322),
  'ergas': 61.20926939757675,
  'ergas_ci95': (50.420453160983925, 73.76507244067639),
  'cc': 0.17555347317130307,
  'cc_ci95': (0.06557607092077275, 0.261097385549125),
  'by_class': {'agri': {'psnr': 11.36272872435388,
    'ssim': 0.1602908190679255,
    'lpips': 0.6960109612689569,
    'sam': 8.096824625363716,
    'ergas': 66.20743374641125,
    'cc': -0.05817233744833407},
   'barrenland': {'psnr': 13.418301276287876,
    'ssim': 0.05567925352859715,
    'lpips': 0.6759745502133475,
    'sam': 4.2547783494371725,
    'ergas': 44.52290078593354,
    'cc': 0.2511620960

In [20]:
from evaluate import evaluate_experiments
evaluate_experiments(leak_audit=True)   # per-patch vs per-scene split, same held-out images


[16:25:12] ======================================================================
[16:25:12] PREFLIGHT  cache=/content/sar_fyp_cache  hf_repo=AliMusaRizvi/sar-to-optical-diffusion  token_set=True
[16:25:12]   GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
[16:25:12] ======================================================================
[16:25:12] data_meta synced from hf://AliMusaRizvi/sar-to-optical-diffusion: ['index.json', 'split_scene.json', 'split_patch.json', 'split_leak_eval.json', 'split_scene_curated.json', 'color_stats.json', 'latent_scale.json', 'precheck_results.json']


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[16:25:13] skip baseline: no local checkpoint yet
[16:25:13] skip patchsplit: no local checkpoint yet


AssertionError: none of the requested experiments have a checkpoint yet

## Not included in this notebook (yet)
The supplementary experiments (test-time tricks, pixel-space baseline, LoRA/adversarial
fine-tune, distillation, quantization) from the earlier round aren't converted to this
no-argparse, direct-function-call pattern here, to keep this notebook's core path solid rather
than spreading thin across everything at once. Ask and I'll convert those the same way next.
